# SmolVLA × LIBERO-plus Spatial Fine-tuning ─ 最終課題教材 (Basic)

## この教材の位置付け

Vision-Language-Action (VLA) モデルを **PyTorch で一から自作** し、実ロボット環境で
学習・評価する一連のワークフローを体験するための最終課題教材です。

具体的には SmolVLA (HuggingFace 2025) の**主要概念** (VLM 凍結 + Flow Matching + action chunking) を取り入れた、**教育用の簡略 VLA** を実装します。公式コードの完全再実装ではなく、学習しやすさを優先した最小限の構成です (公式実装との差異は section 8.5 参照):

1. **Vision-Language 部** (SmolVLM2-500M) を事前学習済み重みで凍結してロード
2. **Action 部** (Flow Matching ベースの transformer) を自作して from-scratch で学習
3. LIBERO-plus 環境で **成功率を測定**、rollout の **動画** を出力
4. 学習条件・結果を **提出用 JSON** にまとめる

---

## 🎯 課題として提出するもの (Basic)

**穴埋め箇所を全部埋めて notebook を上から下まで実行すれば `submission.json` が生成されます。**

`section 8-9` の中に `TODO 1` 〜 `TODO 4` の 4 箇所の**穴埋め問題**があります (`raise NotImplementedError(...)` で明示)。全部正しく埋めないとエラーで止まって notebook が最後まで走らず、提出用 JSON も生成できません。


### <font color="red">※課題提出が完了するまで穴埋め箇所以外の編集は行わないでください</font>

| # | 場所 | 実装するもの | ヒント (数式) |
|---|---|---|---|
| TODO 1 | 8.1 `SinusoidalTimeEmbedding.forward` | 時刻と周波数の外積 | $\text{args}[b, i] = t[b] \cdot \text{freqs}[i]$ |
| TODO 2 | 8.5 `SmolVLA.compute_loss` | 線形パス上の中間点 $x_t$ | $x_t = (1-t)\,x_0 + t\,a$ |
| TODO 3 | 8.5 `SmolVLA.compute_loss` | 真の速度 $v^*$ | $v^* = a - x_0$ |
| TODO 4 | 8.5 `SmolVLA.sample_actions` | Euler 1 歩 | $x \leftarrow x + v \cdot dt$ |

すべて **1 行で書ける短い数式** です。`raise` の直後に `... = ??` のプレースホルダがあるので、`raise` の行を消して `??` の部分を埋めてください。

### 提出物 (Basic)

`section 11` を実行すると **`submission.json` がブラウザにダウンロード**されます (Drive にもバックアップ):

- **`submission.json`** ← ★提出物はこの 1 ファイルだけ★
- `submission.md` / `pip_freeze.txt` — 参照用 (提出不要)

Section 11 のセルを実行すると **自動的にブラウザにダウンロード**され、Drive にもバックアップされます。

**Basic の目的は "実装が動くこと"** であって成績勝負ではありません。**成功率は判定には使いません** — 教材版は from-scratch で少量学習するので基本的に**失敗前提**です (**成功率 0% でも合格**)。 数値でスコアを競いたい / 工夫を評価されたい人は Advanced 用の別 notebook (`smolvla_libero_plus_spatial_advanced.ipynb`) をどうぞ。

---

## 実行環境 (Colab)

本ノートブックは **Google Colab 前提**です。

- 作業ディレクトリは `/content/workdir` (実行速度優先で ephemeral)
- 成果物は **Google Drive の `MyDrive/smolvla_task/` に自動コピー** (ランタイム終了で消えないように)

**実行前に必ずやること**:
- **ランタイム → ランタイムのタイプを変更 → GPU** を選択 (T4 以上)
- Section 1 で **Drive のマウント許可**を求められるので承認する

**Colab の注意**:
- LIBERO-plus assets (6GB) の DL/展開に時間がかかるので、実行前に接続が安定しているか確認
- 連続実行時間の制限があるので、途中で切れないようたまにセルを触っておく
- Drive の空き容量が不十分だとバックアップに失敗するので、`MyDrive` の残量を事前に確認

---

## 使うデータと環境

学習・評価とも [LIBERO-plus](https://github.com/sylvestf/LIBERO-plus) がベース。

### 学習データ (train)

- **HuggingFace dataset**: [`lerobot/libero_plus`](https://huggingface.co/datasets/lerobot/libero_plus)
- **タスク**: LIBERO-Spatial の 10 タスク (すべて「黒いボウルをプレートに置く」バリエーション)
- **エピソード数**: 10 タスク × 5 episode = **50 episode** (section 7 で抽出)

### 評価環境 (eval)

- **シミュレータ**: `LiberoEnv` (LeRobot 経由の LIBERO-plus 摂動シム)
- **タスク**: LIBERO-Spatial 10 タスクから 2 タスク × 1 episode を評価 (合計 2 rollout)

### データフロー

| Step | 何をする | ソース |
|---|---|---|
| データ抽出 | HuggingFace `lerobot/libero_plus` から 50 episode を選抜 | section 7 |
| 学習 | PyTorch SmolVLA を from-scratch で学習 | section 9 |
| 評価 | `LiberoEnv` で rollout | section 10 |
| 提出 | **`submission.json` 生成 + ブラウザ DL** | section 11 |

---

## 既定条件 (`section 6` で変更可)

- 学習 episode: LIBERO-Spatial 10 タスク × 各 5 episode
- 学習 step: 300 
- モデルサイズ: transformer_hidden_dim=256, 4 層 (`section 8.5` で調整可、論文サイズも併記)
- 評価: 2 タスク × 1 episode = 2 rollout (自作 SmolVLA)


## 📖 目次と各セクションの役割

上から順に実行します。**★印**が本課題の中核です。

### 【準備フェーズ】環境構築とデータ準備

| # | セクション | 何をやる |
|---|---|---|
| 1 | Colab ランタイム確認 | GPU の有無 / Python バージョンをチェック、`WORKDIR` を設定 |
| 2 | システムパッケージ | apt で `ffmpeg` / `libgl` 等を導入 |
| 3 | LeRobot インストール | v0.6.0 を clone + editable install |
| 4 | HF 取得ヘルパ | 429 リトライ + キャッシュ優先の取得関数を定義 |
| 5 | LIBERO-plus 環境準備 | MuJoCo + robosuite pin install + **assets 6GB DL/展開 (時間かかる)** |
| 6 | 学習・評価条件を設定 | データセット / seed / パス等の共通定数 |
| 7 | Spatial 学習データ選抜 | 10 タスク × 5 episode = 50 episode を等間隔選抜 |

### 【★中核】SmolVLA 実装 → 学習 → 評価

| # | セクション | 何をやる |
|---|---|---|
| **8** | **★ SmolVLA を PyTorch で実装** | **TODO 1〜4 の穴埋め箇所あり** |
| 8.1 | 時刻埋め込み | 連続時刻 t を高次元ベクトルへ (**TODO 1**) |
| 8.2 | Transformer ブロック | Self/Cross attention + MLP |
| 8.3 | Flow Matching Head | 速度場を予測する transformer |
| 8.4 | SmolVLM2 ロード + freeze | 500M VLM を fp16/bf16 でロードして凍結 (**初回は DL あり**) |
| 8.5 | SmolVLA 統合 | VLM + Action head を結合 (**TODO 2, 3, 4**) |
| 8.6 | 学習対象パラメタの確認 | trainable / frozen の内訳を print |
| **9** | **学習** | 50 episode で自作 SmolVLA を学習 |
| 9.1 | LIBERO データセット | `LeRobotDataset` で action chunk 付きサンプルを供給 |
| 9.2 | action 正規化 | 全 action の mean/std を計算 |
| 9.3 | PyTorch 学習ループ | AdamW + warmup+cosine + 勾配クリッピング |
| **10** | **評価と rollout 動画** | LIBERO env で走らせて mp4 化 |
| 10.1 | LIBERO env で rollout | 2 タスク × 1 episode で成功率測定 + frame 収集 |
| 10.2 | 動画に固めて表示 | `av` (PyAV) で mp4 化 → notebook 埋込 |
| **11** | **★ 提出用ログを書き出す** | **ここで `submission.json` 生成** |

Advanced 課題 (LeRobot ルート、25 タスク広域評価) は別ファイル `smolvla_libero_plus_spatial_advanced.ipynb` で扱います。

---

### 💡 注意:
### <font color="red">※Section1-7は環境構築・事前準備であり、実行するだけでOKです</font>
### <font color="red">※Section8の穴埋めを行い、9-11を実行してください</font>
### <font color="red">※課題提出が完了するまで穴埋め箇所以外の編集は行わないでください</font>



## 1. Colabランタイムを確認する

ColabのランタイムをGPUへ変更してから実行してください。

In [ ]:
# ==============================================================
# 環境変数と実行前チェック + Google Drive マウント
# ==============================================================
import importlib
import importlib.metadata
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch

# --- Google Drive をマウント (成果物の永続化) --------------------
# Colab の /content/ 配下はランタイム終了で消えるので、submission.json や
# 学習済モデルは Drive にも保存できるようにマウントしておく.
# 認証ポップアップが出るのでブラウザで許可すること (初回のみ).
from google.colab import drive

drive.mount("/content/drive")

# Drive 上に成果物をコピーする場所. `MyDrive/smolvla_task/` 配下.
DRIVE_BACKUP_DIR = Path("/content/drive/MyDrive/smolvla_task")
DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)
print(f"DRIVE_BACKUP_DIR: {DRIVE_BACKUP_DIR}")

# --- 作業ディレクトリ ------------------------------------------
# 実行速度重視で /content/ (ephemeral) に置く. Drive 直で作業すると
# 45 万小ファイルの展開などが遅くなる. 最終成果物だけ Drive にコピーする方針.
WORKDIR = Path("/content/workdir")
WORKDIR.mkdir(parents=True, exist_ok=True)
print(f"WORKDIR         : {WORKDIR}")

# --- Hugging Face Hub のノイズ抑制 ------------------------------
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["DIFFUSERS_VERBOSITY"] = "error"
# HF / LeRobot のキャッシュを WORKDIR 配下へ固定
os.environ["HF_HOME"] = str(WORKDIR / "hf_cache")
os.environ["HF_LEROBOT_HOME"] = str(WORKDIR / "lerobot_cache")
# 長時間学習で起こりやすい断片化 OOM を回避
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# LeRobot v0.6.0 は Python 3.12 以上が必須
if sys.version_info < (3, 12):
    raise RuntimeError("Python 3.12以上が必要です。")

# GPU 必須
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPUランタイムを選択してください "
        "(Colab メニュー: ランタイム → ランタイムのタイプを変更 → GPU)"
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. システムパッケージを準備する

LeRobot、動画デコード、MuJoCoで必要になるパッケージを導入します。

In [ ]:
# ==============================================================
# サブプロセス実行ヘルパ + apt でシステムパッケージ導入
# ==============================================================


# 外部コマンドを stdout/stderr をまとめて捕捉しつつ実行する共通ヘルパ:
#   - 成功時はログを出さず notebook を汚さない
#   - 失敗時は末尾 6000 文字だけを例外に載せて可視化する
#   - stderr は stdout に統合して前後関係を保つ
def run_quiet(
    command: list[str],
    *,
    check: bool = True,
) -> subprocess.CompletedProcess:
    result = subprocess.run(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )

    if check and result.returncode != 0:
        raise RuntimeError(result.stdout[-6000:])

    return result


# 長時間コマンド用: 1 行受信するたびに print する streaming 版。
# 進捗が見えない = 無限ループに見える問題を避けるためのヘルパ。
# - prefix を付けて何のコマンドの出力か分かるようにする
# - 直近ログ 200 行をリングバッファで保持し、失敗時ダンプに使う
def run_streaming(
    command: list[str],
    *,
    prefix: str = "",
    check: bool = True,
) -> int:
    from collections import deque

    process = subprocess.Popen(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    recent: deque[str] = deque(maxlen=200)
    assert process.stdout is not None
    for raw in process.stdout:
        line = raw.rstrip("\n")
        recent.append(line)
        print(f"{prefix}{line}", flush=True)
    return_code = process.wait()
    if check and return_code != 0:
        raise RuntimeError(
            "\n".join(recent) or f"command failed: {command}"
        )
    return return_code


# 長時間コマンド用 (pip install 等の出力が数百行になるケース):
# 直近 tail 行だけを ipywidgets の HTML box で "上書き更新" 表示する.
# タイムライン全部を表示すると notebook が肥大化するので、末尾だけ見せる.
def run_streaming_rolling(
    command: list[str],
    *,
    prefix: str = "",
    tail: int = 10,
    check: bool = True,
) -> int:
    from collections import deque
    import ipywidgets as widgets
    from IPython.display import display

    recent: deque[str] = deque(maxlen=tail)
    _widget = widgets.HTML(
        value=(
            "<pre style='margin:0;font-size:11px;color:#666;"
            "line-height:1.3'>(starting...)</pre>"
        ),
        layout=widgets.Layout(margin="4px 0px 4px 32px"),
    )
    display(_widget)

    process = subprocess.Popen(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    assert process.stdout is not None
    _count = 0
    for raw in process.stdout:
        line = raw.rstrip("\n")
        recent.append(line)
        _count += 1
        # 毎行更新すると UI が重いので 5 行ごとに widget を書換
        if _count % 5 == 0:
            _widget.value = (
                "<pre style='margin:0;font-size:11px;color:#666;"
                "line-height:1.3'>"
                + "\n".join(f"  {prefix}{l}" for l in recent)
                + "</pre>"
            )
    # 最終行まで反映
    _widget.value = (
        "<pre style='margin:0;font-size:11px;color:#666;line-height:1.3'>"
        + "\n".join(f"  {prefix}{l}" for l in recent)
        + "\n  ✅ done ({_count} lines total)".replace("{_count}", str(_count))
        + "</pre>"
    )
    return_code = process.wait()
    if check and return_code != 0:
        raise RuntimeError(
            "\n".join(recent) or f"command failed: {command}"
        )
    return return_code


# apt キャッシュ更新
run_quiet(["sudo", "apt-get", "update", "-qq"])
# 依存パッケージ内訳:
#   ffmpeg                : LeRobot の動画デコード
#   git / unzip           : このノートブックで clone / 展開に使用
#   libgl1 / libglib2.0-0 : OpenGL 系（MuJoCo/robosuite の描画）
#   libsm6 / libxext6     : X11 系ランタイム（描画ライブラリの依存）
#   libexpat1             : XML パーサ（MuJoCo の XML モデル読込）
#   libfontconfig1-dev    : matplotlib 用フォント設定
#   libmagickwand-dev     : Wand（ImageMagick binding）が LIBERO 経由で必要
run_quiet(
    [
        "sudo", "apt-get",
        "install",
        "-y",
        "-qq",
        "ffmpeg",
        "git",
        "unzip",
        "libgl1",
        "libglib2.0-0",
        "libsm6",
        "libxext6",
        "libexpat1",
        "libfontconfig1-dev",
        "libmagickwand-dev",
    ]
)

print("System packages ready.")

## 3. LeRobotをインストールする

LeRobot `v0.6.0`を使用します。
ColabでのLoRA学習に必要な互換性調整もこのセルで適用します。

In [ ]:
# ==============================================================
# LeRobot v0.6.0 の editable install と互換性パッチ
# ==============================================================
# 手順:
#   1. 既存 install の除去（torchao も同時に。SmolVLA 非対応のため）
#   2. 固定 tag の LeRobot を shallow clone（進捗を stream）
#   3. bf16 非対応 GPU なら SmolVLM を fp16 に書換
#   4. lerobot_train.py の冗長ログと tqdm を軽微パッチで抑制
#   5. editable install（training + smolvla + peft の extras）— 出力を stream
#   6. torchao を再度アンインストール（依存で復活するケースの保険）
#   7. sys.modules と sys.path を掃除して clone した src を優先させる
LEROBOT_TAG = "v0.6.0"
LEROBOT_DIR = WORKDIR / "lerobot"
LEROBOT_SRC = LEROBOT_DIR / "src"

# --- 0. 安全ガード: torch のバージョン整合性チェック ------------
# 前回のセッションで torch を import 済みで、その後 pip install が
# torch を更新すると、C 拡張(.so)は旧版・Python 側は新版という
# 不整合状態になり "cannot import name '_EvalFrameOverride'" 等が発生する。
# 事前にディスク上と in-memory の torch バージョンを照合し、
# 差があればカーネル再起動を促す。
# torch.__version__ / importlib.metadata.version("torch") は環境によって
# '2.11.0+cu128' のように CUDA タグ (local version identifier) が付いたり
# 付かなかったりする. 両側から '+' 以降を落とした PEP 440 の public version だけで比較.
def _strip_local_version(v: str) -> str:
    return v.split("+", 1)[0]

_torch_version_in_memory = _strip_local_version(torch.__version__)
try:
    _torch_dist_version = _strip_local_version(
        importlib.metadata.version("torch")
    )
except importlib.metadata.PackageNotFoundError:
    _torch_dist_version = None

if _torch_dist_version and _torch_dist_version != _torch_version_in_memory:
    raise RuntimeError(
        f"torch mismatch: in-memory={_torch_version_in_memory} "
        f"on-disk={_torch_dist_version}. "
        "カーネルを再起動してこのセルを最初から実行し直してください。"
    )

# --- 1. 既存 install を除去（再実行安全性のため） ----------------
run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "lerobot",
        "torchao",
    ],
    check=False,
)

shutil.rmtree(LEROBOT_DIR, ignore_errors=True)

# --- 2. LeRobot を shallow clone（進捗を stream 表示） -----------
# ネットワークが遅いと数分かかるが、出力が多いので直近 10 行だけ表示
print("Cloning LeRobot...", flush=True)
run_streaming_rolling(
    [
        "git",
        "clone",
        "--progress",
        "--depth",
        "1",
        "--branch",
        LEROBOT_TAG,
        "https://github.com/huggingface/lerobot.git",
        str(LEROBOT_DIR),
    ],
    prefix="[git] ",
    tail=10,
)

# --- 3. dtype 互換性パッチ -------------------------------------
# T4/V100 系など bf16 未対応 GPU では bf16 で load すると壊れるので fp16 に置換
smolvlm_source = (
    LEROBOT_SRC
    / "lerobot"
    / "policies"
    / "smolvla"
    / "smolvlm_with_expert.py"
)

if not torch.cuda.is_bf16_supported():
    source = smolvlm_source.read_text(encoding="utf-8")
    source = source.replace(
        'torch_dtype="bfloat16",',
        'torch_dtype="float16",',
        1,
    )
    smolvlm_source.write_text(
        source,
        encoding="utf-8",
    )

# --- 4. lerobot_train.py の軽微パッチ ---------------------------
# 学習 subprocess のログを絞る:
#   - 設定 pformat の INFO を DEBUG に落とす
#   - inside_slurm() 判定に依らず tqdm を強制 disable
train_script = (
    LEROBOT_SRC
    / "lerobot"
    / "scripts"
    / "lerobot_train.py"
)
source = train_script.read_text(encoding="utf-8")
source = source.replace(
    "logging.info(pformat(cfg.to_dict()))",
    "logging.debug(pformat(cfg.to_dict()))",
    1,
)
source = source.replace(
    "disable=inside_slurm(),",
    "disable=True,",
    1,
)
train_script.write_text(
    source,
    encoding="utf-8",
)

# --- 5. editable install（extras 込み） ------------------------
# --upgrade を付けない: torch を含む依存が既に満たされていれば
# 何も触らせない（在庫の C 拡張との不整合を避けるため）。
# 依存解決とビルドで 5〜20 分かかる. pip の出力は数百行になるので rolling 表示 (直近 10 行だけ).
print("Installing LeRobot (this may take a while)...", flush=True)
run_streaming_rolling(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--progress-bar",
        "off",
        "-e",
        f"{LEROBOT_DIR}[training,smolvla,peft]",
    ],
    prefix="[pip] ",
    tail=10,
)

# --- 5.5. インストール後 torch が動いていないことを再確認 --------
_torch_dist_version_after = _strip_local_version(
    importlib.metadata.version("torch")
)
if _torch_dist_version_after != _torch_version_in_memory:
    raise RuntimeError(
        f"pip install が torch を {_torch_version_in_memory} → "
        f"{_torch_dist_version_after} に変更しました。"
        "カーネルを再起動してこのセルを最初から実行し直してください。"
    )

# --- 6. torchao 再アンインストール ------------------------------
# 上の install 中に依存として復活することがあるため保険としてもう一度消す
run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "torchao",
    ],
    check=False,
)

# --- 7. sys.modules と sys.path の掃除 --------------------------
# 前回セル実行時に読まれた古い lerobot/torchao モジュールを追い出す
for module_name in list(sys.modules):
    if (
        module_name == "lerobot"
        or module_name.startswith("lerobot.")
        or module_name == "torchao"
        or module_name.startswith("torchao.")
    ):
        del sys.modules[module_name]

# clone した src を最優先で解決させるため sys.path を差し替える
sys.path = [
    item
    for item in sys.path
    if item not in {
        str(LEROBOT_DIR),
        str(LEROBOT_SRC),
    }
]
sys.path.insert(0, str(LEROBOT_SRC))
importlib.invalidate_caches()

# torchao が完全に消えたかチェック。残っていると SmolVLA が起動時に落ちる
try:
    importlib.metadata.version("torchao")
except importlib.metadata.PackageNotFoundError:
    pass
else:
    raise RuntimeError("torchaoの削除に失敗しました。")

# 実際に import して、read 先が今 clone した src 配下であることを保証
# 初回 import は torch/CUDA 初期化で 30〜60 秒かかる場合がある
print("Importing lerobot (torch init may take up to a minute)...", flush=True)
import lerobot
import peft

if (
    LEROBOT_SRC.resolve()
    not in Path(lerobot.__file__).resolve().parents
):
    raise RuntimeError("LeRobotの読込先が正しくありません。")

print("LeRobot ready.")

## 4. 公開ファイルの取得処理を用意する

キャッシュを優先し、匿名アクセスの制限時は自動的に再試行します。

In [ ]:
# ==============================================================
# Hugging Face 取得のリトライ + キャッシュ優先ラッパ
# ==============================================================
# 匿名アクセスは 429 (Too Many Requests) を食らいやすいので指数バックオフを
# 実装し、既存キャッシュがあれば実 DL を回避する。
import random
import time
from collections.abc import Callable
from typing import TypeVar

import httpx
from huggingface_hub import snapshot_download
from huggingface_hub.errors import (
    HfHubHTTPError,
    LocalEntryNotFoundError,
)

T = TypeVar("T")


# 429 のみ最大 6 回リトライ。それ以外のエラーは即再送出する。
def run_hf_with_retry(
    operation: Callable[[], T],
) -> T:
    last_error: BaseException | None = None

    for attempt in range(6):
        try:
            return operation()
        except (
            HfHubHTTPError,
            httpx.HTTPStatusError,
        ) as error:
            last_error = error
            response = getattr(error, "response", None)
            status = getattr(response, "status_code", None)

            # 429 以外はリトライしない（404 などを叩き続けない）
            if status != 429 and "429" not in str(error):
                raise

            if attempt == 5:
                break

            # サーバが Retry-After を返せば尊重、無ければ指数バックオフ
            headers = getattr(response, "headers", {}) or {}
            try:
                delay = float(
                    headers.get("Retry-After", 15)
                ) + 1
            except (TypeError, ValueError):
                delay = min(
                    120,
                    15 * (2**attempt) + random.random(),
                )

            time.sleep(delay)

    raise RuntimeError(
        "Hugging Faceからの取得に失敗しました。"
    ) from last_error


# まず local_files_only でキャッシュ確認、無ければ実 DL する。
# 大きなモデル/データセットを何度も再取得しないための最適化。
def cached_or_downloaded_snapshot(
    repo_id: str,
    revision: str,
    *,
    allow_patterns: list[str] | None = None,
    ignore_patterns: list[str] | None = None,
) -> Path:
    try:
        # キャッシュヒット時はネットワークに触れず即返す
        return Path(
            snapshot_download(
                repo_id=repo_id,
                revision=revision,
                token=False,
                allow_patterns=allow_patterns,
                ignore_patterns=ignore_patterns,
                local_files_only=True,
            )
        )
    except (
        LocalEntryNotFoundError,
        FileNotFoundError,
    ):
        # キャッシュミス → 429 リトライ付きで実 DL
        # max_workers=1 でサーバに優しくアクセス（429 回避）
        return Path(
            run_hf_with_retry(
                lambda: snapshot_download(
                    repo_id=repo_id,
                    revision=revision,
                    token=False,
                    allow_patterns=allow_patterns,
                    ignore_patterns=ignore_patterns,
                    max_workers=1,
                )
            )
        )

## 5. LIBERO-plus 評価環境を準備する

MuJoCo / robosuite / LIBERO-plus fork をインストールし、評価用アセットを展開します。

**assets.zip (約 6GB) のダウンロード + 展開があるため、このセルは時間がかかります。**

In [ ]:
# ==============================================================
# LIBERO-plus 評価環境をセットアップ
# ==============================================================
# 手順:
#   1. MuJoCo/robosuite 系の依存を pin 付きで install
#   2. LIBERO-plus fork を固定 SHA で clone/checkout
#   3. assets.zip を HF から取得して展開
#   4. ~/.libero/config.yaml を書いてアセット位置を教える
#   5. lerobot_eval.py に進捗表示用のパッチを当てる
#   6. sys.path / sys.modules を掃除して fork 側を優先ロード
from huggingface_hub import hf_hub_download
import time

LIBERO_PLUS_SHA = "4976dc3"
LIBERO_PLUS_DIR = WORKDIR / "LIBERO-plus"
LIBERO_PLUS_PACKAGE_ROOT = (
    LIBERO_PLUS_DIR / "libero" / "libero"
)
# assets (45 万小ファイル 6GB) の保存先. /content/ 配下なので展開は高速.
LIBERO_PLUS_ASSETS_DIR = WORKDIR / "libero_plus_assets" / "assets"

# Colab のヘッドレス GPU 環境では EGL バックエンドで MuJoCo を描画
os.environ["MUJOCO_GL"] = "egl"

# --- 1. 既存 install を除去（再実行安全性） --------------------
run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "hf-libero",
        "libero",
        "robosuite",
    ],
    check=False,
)

# --- 2. LIBERO-plus 依存の pin 付き install --------------------
# バージョンを揃えないと robosuite / mujoco の API 差でクラッシュするため厳格に固定
run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "robosuite==1.4.1",  # LIBERO-plus が期待する固定版
        "bddl==1.0.1",
        "easydict==1.13",
        "mujoco==3.7.0",
        "matplotlib==3.10.8",
        "Wand==0.6.13",
        "scikit-image==0.25.2",
        "gym==0.26.2",
        "future",  # bddl 1.0.1 が依存宣言していない隠れ依存
    ]
)

# robosuite が期待どおりの版でロードされることを担保
if (
    importlib.metadata.version("robosuite")
    != "1.4.1"
):
    raise RuntimeError(
        "robosuite 1.4.1 is required."
    )

# --- 3. LIBERO-plus fork を clone -----------------------------
if not (LIBERO_PLUS_DIR / ".git").is_dir():
    shutil.rmtree(
        LIBERO_PLUS_DIR,
        ignore_errors=True,
    )
    run_quiet(
        [
            "git",
            "clone",
            "--quiet",
            "https://github.com/sylvestf/LIBERO-plus.git",
            str(LIBERO_PLUS_DIR),
        ]
    )

# 固定 SHA へ checkout。既存 clone に対しても再実行安全になるよう、
# checkout 失敗時のみ fetch でリカバリする
checkout = run_quiet(
    [
        "git",
        "-C",
        str(LIBERO_PLUS_DIR),
        "checkout",
        "--quiet",
        LIBERO_PLUS_SHA,
    ],
    check=False,
)

if checkout.returncode != 0:
    # SHA がローカルに無い（shallow clone 等）なら fetch してから checkout
    run_quiet(
        [
            "git",
            "-C",
            str(LIBERO_PLUS_DIR),
            "fetch",
            "--quiet",
            "--depth",
            "1",
            "origin",
            LIBERO_PLUS_SHA,
        ]
    )
    run_quiet(
        [
            "git",
            "-C",
            str(LIBERO_PLUS_DIR),
            "checkout",
            "--quiet",
            LIBERO_PLUS_SHA,
        ]
    )

# fork を editable install。--no-deps で pin 済みの依存を上書きされないようにする
run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-deps",
        "-e",
        str(LIBERO_PLUS_DIR),
    ]
)

# --- 4. assets.zip を HF から DL・展開 -------------------------
# 大きな asset は fork 側に含まれないので Sylvest/LIBERO-plus から取得
if not LIBERO_PLUS_ASSETS_DIR.is_dir():
    assets_root = WORKDIR / "libero_plus_assets"
    assets_root.mkdir(parents=True, exist_ok=True)

    # ---- Drive キャッシュ戦略 ----
    # 優先度: (1) tar.gz キャッシュ (最速)  > (2) zip キャッシュ  > (3) HF DL
    # tar.gz を Drive に持っておくと、次回セッションから展開まで含めて数分で終わる.
    _drive_tar = DRIVE_BACKUP_DIR / "libero_plus_assets.tar.gz"
    _drive_zip = DRIVE_BACKUP_DIR / "libero_plus_assets.zip"

    if _drive_tar.is_file():
        # ---- 最速パス: tar.gz を Drive から取ってきて展開 ----
        print(
            f"Drive の tar.gz キャッシュから復元中 (unzip 不要): {_drive_tar}",
            flush=True,
        )
        _t0 = time.time()
        import tarfile
        from tqdm.auto import tqdm as _tqdm_tar

        # tar.gz は zip と違って中央ディレクトリを持たないため、
        # 通常はストリーム走査しないとファイル総数が分からない.
        # そこで前回の tar 作成時に .count サイドカーへ件数を書き出しておき、
        # あればそれを使って tqdm の total を即座に埋める.
        _drive_tar_count = _drive_tar.with_suffix(_drive_tar.suffix + ".count")
        _tar_total = None
        if _drive_tar_count.is_file():
            try:
                _tar_total = int(_drive_tar_count.read_text().strip())
                print(f"サイドカーから件数を取得: {_tar_total} files", flush=True)
            except Exception:
                _tar_total = None
        if _tar_total is None:
            # サイドカー無し (旧キャッシュ): getmembers() で 1 度走査する.
            # data は展開しないのでフル展開よりは速いが、それでも数十秒待つ.
            print("件数を数えています (初回のみ、少し時間かかる)...", flush=True)
            with tarfile.open(_drive_tar, "r:gz") as _tf:
                _tar_total = sum(1 for _ in _tf)

        with tarfile.open(_drive_tar, "r:gz") as _tf:
            _pbar_tx = _tqdm_tar(total=_tar_total, desc="tar 展開", unit="files")
            while True:
                _m = _tf.next()
                if _m is None:
                    break
                _tf.extract(_m, assets_root)
                _pbar_tx.update(1)
            _pbar_tx.close()
        print(f"tar 展開完了 (in {time.time()-_t0:.0f}s)", flush=True)

        # 古い zip キャッシュを削除 (Drive 容量節約)
        if _drive_zip.is_file():
            try:
                _drive_zip.unlink()
                print(f"古い zip キャッシュを削除: {_drive_zip}", flush=True)
            except Exception:
                pass
        # → LIBERO_PLUS_ASSETS_DIR が存在するはず. 以下 DL/unzip はスキップ.
        _skip_dl_unzip = True
    else:
        _skip_dl_unzip = False

if not LIBERO_PLUS_ASSETS_DIR.is_dir() and not _skip_dl_unzip:
    # ---- 中速パス: zip キャッシュから復元 or HF DL ----
    local_zip = assets_root / "assets.zip"

    if _drive_zip.is_file() and not local_zip.is_file():
        print(f"Drive キャッシュから assets.zip を復元: {_drive_zip}", flush=True)
        shutil.copy2(_drive_zip, local_zip)

    if local_zip.is_file():
        archive_path = local_zip
        print(f"既存の assets.zip を使用: {archive_path}", flush=True)
    else:
        print("HF から assets.zip をダウンロード中 (6GB, 時間かかる)...", flush=True)
        # ---- HF DL 中は progress bar を有効化 ----
        # env-var の pop は huggingface_hub が import 時に disabled をキャッシュするため効かない.
        # 公式 API enable_progress_bars() / disable_progress_bars() で明示的に切り替える.
        from huggingface_hub.utils import (
            enable_progress_bars,
            disable_progress_bars,
        )
        enable_progress_bars()
        try:
            _t0 = time.time()
            archive_path = Path(
                run_hf_with_retry(
                    lambda: hf_hub_download(
                        repo_id="Sylvest/LIBERO-plus",
                        repo_type="dataset",
                        filename="assets.zip",
                        local_dir=assets_root,
                        token=False,
                    )
                )
            )
            _dl_sec = time.time() - _t0
        finally:
            disable_progress_bars()

        _size_mb = archive_path.stat().st_size / 1e6
        print(
            f"HF DL 完了: {_size_mb:.0f} MB in {_dl_sec:.0f}s "
            f"({_size_mb / max(_dl_sec, 1):.1f} MB/s)",
            flush=True,
        )
        # 注: この段階では Drive に zip を書かない.
        # 展開後に tar.gz を作って Drive に置く (下の方の処理) 方が
        # 次セッションからの復元も速くて経済的.
    extract_dir = assets_root / "extract"

    shutil.rmtree(extract_dir, ignore_errors=True)
    extract_dir.mkdir(parents=True, exist_ok=True)

    # 展開は Python の zipfile + tqdm progress bar で行う.
    # ・全体進捗は tqdm バーで見せる (ETA / files/sec 含む).
    # ・直近 10 ファイル名を ipywidgets の HTML box で "上書き更新" する
    #   (subprocess の unzip -o だと 45 万行の出力になって notebook が固まるため).
    import zipfile
    import ipywidgets as widgets
    from IPython.display import display
    from tqdm.auto import tqdm
    from collections import deque

    print(f"Extracting {archive_path.name} to {extract_dir}...", flush=True)

    # 直近展開ファイル 10 件を表示するウィジェット
    _recent_paths: deque[str] = deque(maxlen=10)
    _recent_widget = widgets.HTML(
        value=(
            "<pre style='margin:0;font-size:11px;color:#666;"
            "line-height:1.3'>(starting...)</pre>"
        ),
        layout=widgets.Layout(margin="4px 0px 4px 32px"),
    )
    display(_recent_widget)

    _t0 = time.time()
    with zipfile.ZipFile(archive_path) as _zf:
        _members = _zf.infolist()
        _pbar = tqdm(_members, desc="unzip", unit="files")
        for _i, _m in enumerate(_pbar):
            _recent_paths.append(_m.filename)
            _zf.extract(_m, extract_dir)
            # 500 ファイルごとに widget を上書き (毎回だと重い)
            if _i % 500 == 0:
                _recent_widget.value = (
                    "<pre style='margin:0;font-size:11px;color:#666;"
                    "line-height:1.3'>"
                    + "\n".join(f"  {p}" for p in _recent_paths)
                    + "</pre>"
                )
        # 最終状態
        _recent_widget.value = (
            "<pre style='margin:0;font-size:11px;color:#666;"
            "line-height:1.3'>"
            + f"  ✅ done: {len(_members)} files extracted"
            + "</pre>"
        )
    print(f"unzip 完了 (in {time.time()-_t0:.0f}s)", flush=True)

    # zip 内のディレクトリ階層は変わりうるので rglob で assets/ を探す。
    # ネストが浅いものを優先（本物の assets ルートに近いと想定）
    candidates = sorted(
        [
            path
            for path in extract_dir.rglob("assets")
            if path.is_dir()
        ],
        key=lambda path: len(path.parts),
    )

    if not candidates:
        raise FileNotFoundError(
            "LIBERO-plus assets not found."
        )

    LIBERO_PLUS_ASSETS_DIR.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    # 展開した本物の assets を fork 内の期待位置へ移動
    shutil.move(
        str(candidates[0]),
        str(LIBERO_PLUS_ASSETS_DIR),
    )
    # assets の移動が終わったので extract テンポラリだけ削除。
    shutil.rmtree(extract_dir, ignore_errors=True)

    # ---- 次セッション用に tar.gz を Drive にキャッシュ ----
    # zip + unzip のフルパスは重いので、展開後の assets を tar.gz 化して Drive に保存.
    # 次回セッションでは tar 展開だけになり、数分で復元できる.
    try:
        _t0 = time.time()
        print(
            f"Drive に tar.gz キャッシュを作成中...",
            flush=True,
        )
        import tarfile
        from tqdm.auto import tqdm as _tqdm_tar_c

        # 作成時は事前にファイル数をカウントできるので tqdm の total を渡せる
        _entries = list(LIBERO_PLUS_ASSETS_DIR.rglob("*"))
        _total_entries = len(_entries) + 1  # +1: root dir 自身
        _pbar_tc = _tqdm_tar_c(total=_total_entries, desc="tar 作成", unit="files")

        def _tar_filter(tarinfo):
            _pbar_tc.update(1)
            return tarinfo

        with tarfile.open(_drive_tar, "w:gz", compresslevel=1) as _tf:
            _tf.add(LIBERO_PLUS_ASSETS_DIR, arcname="assets", filter=_tar_filter)
        _pbar_tc.close()
        # 次回復元時に tqdm の total を即座に出せるよう、件数サイドカーを保存.
        _drive_tar.with_suffix(_drive_tar.suffix + ".count").write_text(
            str(_total_entries) + "\n"
        )
        print(f"tar.gz キャッシュ完了 (in {time.time()-_t0:.0f}s)", flush=True)
        # 古い zip キャッシュを削除 (Drive 容量節約)
        if _drive_zip.is_file():
            try:
                _drive_zip.unlink()
                print(f"旧 zip キャッシュを削除: {_drive_zip}", flush=True)
            except Exception:
                pass
    except Exception as _e:
        print(f"tar.gz キャッシュ作成失敗 (無視して継続): {_e}", flush=True)

# --- 5. ~/.libero/config.yaml を書く --------------------------
# LIBERO は起動時にこのファイルを読んで assets/bddl/datasets/init_files の
# 場所を解決する。fork 側のパスに向ける
# ---- LIBERO の期待する assets パスへの symlink を作る ----
# LIBERO の robosuite Arena XML 読込は fork 内の相対パス
# ({LIBERO_PLUS_DIR}/libero/libero/assets/...) をハードコードで参照する.
# 一方で本 notebook は 45 万小ファイルの実体を LIBERO_PLUS_ASSETS_DIR に置いてある.
# 両者を symlink で繋ぐことで、LIBERO は "期待の場所" から実体を辿れる.
_libero_expected_assets = LIBERO_PLUS_DIR / "libero" / "libero" / "assets"
if _libero_expected_assets.is_symlink() or _libero_expected_assets.exists():
    if _libero_expected_assets.is_symlink():
        _libero_expected_assets.unlink()
    else:
        shutil.rmtree(_libero_expected_assets, ignore_errors=True)
_libero_expected_assets.parent.mkdir(parents=True, exist_ok=True)
_libero_expected_assets.symlink_to(LIBERO_PLUS_ASSETS_DIR)
print(f"symlink: {_libero_expected_assets} -> {LIBERO_PLUS_ASSETS_DIR}")

libero_config_dir = Path.home() / ".libero"
libero_config_dir.mkdir(
    parents=True,
    exist_ok=True,
)
(libero_config_dir / "config.yaml").write_text(
    "\n".join(
        [
            f"assets: {LIBERO_PLUS_ASSETS_DIR}",
            (
                "bddl_files: "
                f"{LIBERO_PLUS_PACKAGE_ROOT / 'bddl_files'}"
            ),
            (
                "datasets: "
                f"{LIBERO_PLUS_PACKAGE_ROOT.parent / 'datasets'}"
            ),
            (
                "init_states: "
                f"{LIBERO_PLUS_PACKAGE_ROOT / 'init_files'}"
            ),
        ]
    )
    + "\n",
    encoding="utf-8",
)

# --- 6. lerobot_eval.py に進捗表示パッチを当てる -----------------
# デフォルトの評価スクリプトは冗長な log と tqdm を吐くので抑制。
# 代わりに "EVAL_PROGRESS task=x/y episode=a/b" 形式を stdout に流し、
# こちらのノート側でパースして表示する。
eval_script = (
    LEROBOT_SRC
    / "lerobot"
    / "scripts"
    / "lerobot_eval.py"
)
source = eval_script.read_text(encoding="utf-8")

# 冗長な設定 pformat を DEBUG に落とす
source = source.replace(
    "logging.info(pformat(asdict(cfg)))",
    "logging.debug(pformat(asdict(cfg)))",
    1,
)
# 録画時以外に rollout 動画を最大 10 本描画してしまう箇所を 0 に固定
source = source.replace(
    "max_episodes_rendered = 0 if cfg.eval.recording else 10",
    "max_episodes_rendered = 0",
    1,
)
# tqdm を強制無効化
source = source.replace(
    "disable=inside_slurm()",
    "disable=True",
)

# 現在の task/episode 進捗を保持するグローバル辞書をスクリプトへ注入
progress_state = (
    '_EVAL_PROGRESS = {"task_index": 0, "task_total": 0}'
)
if progress_state not in source:
    import_anchor = "from tqdm import trange\n"
    if import_anchor not in source:
        raise RuntimeError(
            "Evaluation progress import anchor not found."
        )
    source = source.replace(
        import_anchor,
        import_anchor + "\n" + progress_state + "\n",
        1,
    )

# task ループの先頭で _EVAL_PROGRESS を更新するコードを差し込む
task_loop_anchor = (
    "        for i, (task_group, task_id, env) "
    "in enumerate(tasks):\n"
)
task_loop_patch = (
    task_loop_anchor
    + '            _EVAL_PROGRESS["task_index"] = i + 1\n'
    + '            _EVAL_PROGRESS["task_total"] = len(tasks)\n'
)
if (
    '_EVAL_PROGRESS["task_index"] = i + 1'
    not in source
):
    if task_loop_anchor not in source:
        raise RuntimeError(
            "Evaluation task-loop anchor not found."
        )
    source = source.replace(
        task_loop_anchor,
        task_loop_patch,
        1,
    )

# episode ループの各周で進捗 1 行を stdout に流す
episode_loop_anchor = "    for batch_ix in progbar:\n"
episode_progress_line = (
    '        print('
    'f"EVAL_PROGRESS '
    "task={_EVAL_PROGRESS['task_index']}/"
    "{_EVAL_PROGRESS['task_total']} "
    'episode={batch_ix + 1}/{n_batches}", '
    "flush=True)\n"
)
if "EVAL_PROGRESS task=" not in source:
    if episode_loop_anchor not in source:
        raise RuntimeError(
            "Evaluation episode-loop anchor not found."
        )
    source = source.replace(
        episode_loop_anchor,
        episode_loop_anchor + episode_progress_line,
        1,
    )

eval_script.write_text(
    source,
    encoding="utf-8",
)

# --- 7. sys.path 掃除と fork のロード検証 ---------------------
# pip install した LIBERO-plus fork が絶対に手前に来るようにする
libero_plus_path = str(LIBERO_PLUS_DIR)
sys.path = [
    item
    for item in sys.path
    if item != libero_plus_path
]
sys.path.insert(0, libero_plus_path)

# 古い LIBERO/robosuite モジュールキャッシュを追い出す
for module_name in list(sys.modules):
    if (
        module_name == "libero"
        or module_name.startswith("libero.")
        or module_name == "robosuite"
        or module_name.startswith("robosuite.")
    ):
        del sys.modules[module_name]

importlib.invalidate_caches()

import libero
from libero.libero import benchmark

# libero パッケージの読込先が fork ディレクトリ配下であることを保証
search_paths = [
    Path(path).resolve()
    for path in getattr(libero, "__path__", [])
]

if not any(
    LIBERO_PLUS_DIR.resolve() in path.parents
    or path == LIBERO_PLUS_DIR.resolve()
    for path in search_paths
):
    raise RuntimeError(
        "LIBERO-plus fork was not loaded."
    )

# benchmark モジュールについても同様に確認
benchmark_path = Path(
    benchmark.__file__
).resolve()

if (
    LIBERO_PLUS_DIR.resolve()
    not in benchmark_path.parents
):
    raise RuntimeError(
        "LIBERO-plus benchmark was not loaded."
    )

print("LIBERO-plus ready.")

## 6. 学習・評価条件を設定する

このセルで定義するのは **両ルート共通のもの** だけ:

- モデル / データセットの repo と revision
- LIBERO-Spatial 10 タスクのリスト
- 学習に使う episode 数と seed
- 出力パス、混合精度

各ルート固有の学習ハイパラは以下:
- PyTorch 版 → `section 9.3` の `TRAIN_STEPS_PY` 等
- LeRobot 版 → `Advanced` の `STEPS` 等

In [ ]:
# ==============================================================
# 学習・評価に使う定数とパスをまとめて定義
# ==============================================================

# --- モデル / データセットの固定 revision -----------------------
# revision を commit hash で固定して再現性を担保
BASE_MODEL_REPO = "lerobot/smolvla_libero_plus"
BASE_MODEL_REVISION = (
    "7bb70aa5bc92b82c9239142775d3a173103567ff"
)

# SmolVLA の Vision-Language backbone（500M パラメータ版）
VLM_REPO = (
    "HuggingFaceTB/SmolVLM2-500M-Video-Instruct"
)

DATASET_REPO = "lerobot/libero_plus"
DATASET_REVISION = (
    "f3f49f426d75030177b18778374005bc12ccd588"
)

# --- LIBERO-Spatial の 10 タスク（学習・評価で共通） -------------
# データセット側の task 名と厳密一致させる必要がある
SPATIAL_TASK_NAMES = [
    "pick up the black bowl from table center and place it on the plate",
    "pick up the black bowl next to the cookie box and place it on the plate",
    "pick up the black bowl next to the plate and place it on the plate",
    "pick up the black bowl next to the ramekin and place it on the plate",
    "pick up the black bowl on the cookie box and place it on the plate",
    "pick up the black bowl on the ramekin and place it on the plate",
    "pick up the black bowl on the stove and place it on the plate",
    "pick up the black bowl on the wooden cabinet and place it on the plate",
    "pick up the black bowl in the top drawer of the wooden cabinet and place it on the plate",
    "pick up the black bowl between the plate and the ramekin and place it on the plate",
]

# --- 学習・評価データ選抜 --------------------------------------
TRAIN_EPISODES_PER_TASK = 5   # 10 tasks × 5 = 50 episodes 選抜
SEED = 42                     # 乱数 seed
EVAL_SEED = 2026              # 評価 seed

# --- 出力パス --------------------------------------------------
OUTPUT_DIR = WORKDIR / "outputs" / "smolvla_libero_plus_spatial_lora"
MERGED_MODEL_DIR = WORKDIR / "smolvla_libero_plus_spatial_lora_merged"
BASELINE_MODEL_DIR = WORKDIR / "smolvla_libero_plus_baseline"

BASE_EVAL_DIR = WORKDIR / "eval" / "base"
FINETUNED_EVAL_DIR = WORKDIR / "eval" / "spatial_lora"
COMPARISON_CSV_PATH = WORKDIR / "libero_spatial_comparison.csv"
MERGED_ZIP_PATH = WORKDIR / "smolvla_libero_plus_spatial_lora_merged.zip"

# --- 混合精度モード --------------------------------------------
# bf16 対応 GPU（A100/H100/L40/RTX40 系など）なら bf16、それ以外は fp16
MIXED_PRECISION = (
    "bf16"
    if torch.cuda.is_bf16_supported()
    else "fp16"
)

## 7. Spatial 学習データを選ぶ

10タスクから各5エピソードを等間隔に選択します。

In [ ]:
# ==============================================================
# LIBERO-Spatial 10 タスクから各 5 episode を等間隔選択
# ==============================================================
# 学習用サブセットを再現可能に構築する。
# データセット側の task 名は表記揺れがありうるので正規化キーで突合し、
# 各 task の episode index から等間隔に 5 個を間引く（合計 50 episode）。
import re
from collections import defaultdict

from lerobot.datasets.dataset_metadata import (
    LeRobotDatasetMetadata,
)


# 小文字化・記号除去・空白正規化で突合用キーを作る。
def normalize_task_name(value: str) -> str:
    value = value.lower().replace("_", " ")
    value = re.sub(r"[^a-z0-9 ]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


# メタデータの tasks 列は str/list どちらもあり得るので単一 str に落とす。
def task_name_from_cell(value) -> str:
    if isinstance(value, str):
        return value

    try:
        if len(value) > 0:
            return str(value[0])
    except TypeError:
        pass

    return str(value)


# episode 群から等間隔に count 個を選ぶ（先頭と末尾を必ず含む）。
def choose_evenly_spaced(
    episode_indices: list[int],
    count: int,
) -> list[int]:
    if len(episode_indices) < count:
        raise ValueError(
            f"Task には episode が {len(episode_indices)} 件しかないため、"
            f"要求数 {count} を満たせません。"
        )
    positions = [
        round(
            index
            * (len(episode_indices) - 1)
            / (count - 1)
        )
        for index in range(count)
    ]

    return [
        episode_indices[position]
        for position in positions
    ]


# データセットのメタデータを取得（429 リトライ付き）
dataset_metadata = run_hf_with_retry(
    lambda: LeRobotDatasetMetadata(
        DATASET_REPO,
        revision=DATASET_REVISION,
    )
)

# 「task 名 → その task に属する episode index リスト」を構築
task_to_episodes: dict[str, list[int]] = defaultdict(list)

for episode_index, task_cell in enumerate(
    dataset_metadata.episodes["tasks"]
):
    task_to_episodes[
        task_name_from_cell(task_cell)
    ].append(int(episode_index))

# 正規化キーから実際のタスク名（表記揺れ含む）を引くための辞書
available_by_normalized = {
    normalize_task_name(task_name): task_name
    for task_name in task_to_episodes
}

# Spatial の 10 タスク各々から 5 episode を選ぶ
selected_by_task: dict[str, list[int]] = {}

for task_name in SPATIAL_TASK_NAMES:
    actual_task = available_by_normalized.get(
        normalize_task_name(task_name)
    )

    if actual_task is None:
        raise RuntimeError(
            f"Spatial task not found: {task_name}"
        )

    selected_by_task[actual_task] = choose_evenly_spaced(
        task_to_episodes[actual_task],
        TRAIN_EPISODES_PER_TASK,
    )

# フラットな episode index のソート済みリストへ集約
EPISODE_INDICES = sorted(
    episode_index
    for episode_indices in selected_by_task.values()
    for episode_index in episode_indices
)

# 期待: 10 tasks × 5 = 50 個
if len(EPISODE_INDICES) != 50:
    raise RuntimeError("Episode selection failed.")

print("Training data: 10 tasks × 5 episodes = 50 episodes")

## 8. SmolVLA を PyTorch で実装する

このセクションが本チュートリアルの中心。**Vision-Language-Action モデル SmolVLA** を、
論文と実装を照らし合わせながら PyTorch で書き下します。

### 参照論文・実装

- **SmolVLA アーキテクチャ** — Shukor et al., "SmolVLA: A Vision-Language-Action Model for Affordable and Efficient Robotics" (HuggingFace, 2025) — <https://huggingface.co/blog/smolvla>
- **Flow Matching (action head の生成モデル)** — Lipman et al., "Flow Matching for Generative Modeling" (ICLR 2023) — <https://arxiv.org/abs/2210.02747>
- **Transformer self/cross-attention** — Vaswani et al., "Attention Is All You Need" (NeurIPS 2017) — <https://arxiv.org/abs/1706.03762>
- **LoRA (Low-Rank Adaptation)** — Hu et al., "LoRA: Low-Rank Adaptation of Large Language Models" (ICLR 2022) — <https://arxiv.org/abs/2106.09685>
- **LIBERO ベンチマーク** — Liu et al., "LIBERO: Benchmarking Knowledge Transfer for Lifelong Robot Learning" (NeurIPS 2023) — <https://arxiv.org/abs/2306.03310>
- **ベース実装** — LeRobot v0.6.0 policies.smolvla — <https://github.com/huggingface/lerobot/tree/v0.6.0/src/lerobot/policies/smolvla>

### 全体構造

VLA モデルは 2 段のパイプライン:

- **1. VLM (frozen)** — 入力: 画像 $x \in \mathbb{R}^{H \times W \times 3}$ + タスク説明 $\tau$ → 出力: $h \in \mathbb{R}^{L \times 960}$
  - SmolVLM2-500M で融合表現に変換 (パラメタは学習しない)
- **2. Action head (Flow Matching Head, trainable)** — 入力: noisy action chunk $x_t$, 時刻 $t$, 条件 $(h,\ \text{state})$ → 出力: 速度場 $v_\theta(x_t, t, h) \in \mathbb{R}^{K \times d_a}$
  - 内部は Transformer (self-attn + VLM への cross-attn + MLP を $N$ 層) と入出力射影
  - Flow Matching (Lipman et al. 2023) で action chunk を生成

$K$ = `chunk_size` (未来 step 数), $d_a$ = action 次元 (LIBERO は 7 = 6-DoF + gripper), $L$ = 画像 patch + text token 数

**要点**: VLM は凍結 (500M params 触らない)、Action head の数 M params だけを学習する。

### Flow Matching の数式 (詳細)

真の行動 $a \in \mathbb{R}^{K \times d_a}$ と Gaussian ノイズ $x_0 \sim \mathcal{N}(0, I)$ に対し、
時刻 $t \in [0, 1]$ の**直線パス**を定義:

$$x_t = (1 - t)\, x_0 + t\, a$$

このパス上の**真の速度**は簡単に:

$$v^*(x_t, t) = \frac{d x_t}{dt} = a - x_0$$

条件 $c$ (画像・言語) を受けた**ネットワーク** $v_\theta(x_t, t, c)$ に、
$v^*$ を回帰させるのが Flow Matching Loss:

$$\mathcal{L}(\theta) = \mathbb{E}_{a \sim \mathcal{D},\ x_0 \sim \mathcal{N},\ t \sim U(0,1)} \left[ \big\| v_\theta(x_t, t, c) - (a - x_0) \big\|^2 \right]$$

**推論** (sampling) は、pure noise $x_0 \sim \mathcal{N}(0, I)$ から Euler 法で ODE を解く:

$$x_{t+\Delta t} = x_t + v_\theta(x_t, t, c) \cdot \Delta t$$

を $t = 0 \to 1$ まで $N$ 段階で進めれば、$x_1 \approx a$ が得られます (通常 $N \in [5, 20]$)。

### なぜ Flow Matching (vs Diffusion)

- **学習が単純**: DDPM は各時刻の noise schedule と分散を扱う必要があるが、FM は「真の速度 = $a - x_0$」だけ
- **推論が速い**: 直線パスなので少ステップ (5〜10) でも精度が出やすい
- **数値的に安定**: 発散しにくい

### 実装のポイント

- **VLM を凍結**: 500M パラメタは学習させない (計算資源と過学習の両面で不利)
- **Action head だけ学習**: Transformer + 入出力射影 + 位置/時刻埋め込みで数 M パラメタ
- **Action head は from-scratch でフル学習**: LoRA は使わない (詳細は 8.6 参照)

以降のセルで、この構造をパーツごとに組み立てます。


### 8.1 時刻埋め込み

Flow Matching は時刻 $t \in [0, 1]$ を条件としてネットワークに渡します。
しかしスカラー $t$ をそのまま入れると微妙な違いが捉えづらいので、**高次元ベクトルに埋め込む**。

Transformer の位置エンコーディング (Vaswani et al., NeurIPS 2017) と同じ発想:

$$\text{PE}(t, 2i) = \sin\left(\frac{t}{10000^{2i/D}}\right), \quad \text{PE}(t, 2i+1) = \cos\left(\frac{t}{10000^{2i/D}}\right)$$

- $D$ = hidden_dim
- $i = 0, 1, \ldots, D/2 - 1$
- **高周波** ($i$ 小) = 細かい差、**低周波** ($i$ 大) = 大局的な位置 を同時に表現できる

**Diffusion 論文でも同じテクニックが使われている** (DDPM, Ho et al. 2020)。


In [ ]:
# ==============================================================
# 8.1 時刻埋め込み (SinusoidalTimeEmbedding)
# ==============================================================
#
# 【目的】 連続時刻 t ∈ [0, 1] をネットワークに条件として渡すため、高次元ベクトルへ埋め込む。
#
# 【数式】  (D = hidden_dim, i = 0..D/2-1)
#     embed(t)[2i]   = sin(t / 10000^(2i/D))
#     embed(t)[2i+1] = cos(t / 10000^(2i/D))
#
# 【背景】
#   Transformer の位置エンコーディング (Vaswani et al. 2017) を「離散位置 → 連続時刻」に拡張したもの。
#   Diffusion モデルでもタイムステップ埋め込みとして常用される (DDPM, Ho et al. 2020)。
#
# 【なぜ高次元に埋め込むか】
#   スカラー t だけだと入力層で `nn.Linear(1, D)` などで拡張することになるが、
#   sin/cos は「異なる周波数を並べる」ことで、大局〜細部を同時に伝えられる。学習も安定。

# section 8 全体で共通の import
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class SinusoidalTimeEmbedding(nn.Module):
    """スカラー時刻 t を hidden_dim 次元のベクトルに埋め込む."""

    def __init__(self, hidden_dim: int):
        super().__init__()
        # sin と cos で半分ずつ埋めるので偶数を要求
        assert hidden_dim % 2 == 0, "hidden_dim は偶数にしてください"
        self.hidden_dim = hidden_dim  # 例: 960 (SmolVLM2 の text hidden size)

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        # -------- 入力/出力の shape 契約 --------
        # 入力 t:      shape = (B,)              batch 内の各サンプルの時刻 (0〜1)
        # 出力:        shape = (B, hidden_dim)   各サンプルの埋め込みベクトル

        half = self.hidden_dim // 2  # 例: 480

        # ---- Step 1: 周波数の等比数列 ----
        # freqs[i] = exp(-log(10000) * i / half) = 1 / 10000^(i/half)
        # 高周波 (freqs[0]=1, 周期 2π) → 低周波 (freqs[half-1]≈1/10000, 周期 20000π)
        freqs = torch.exp(
            -math.log(10000.0)
            * torch.arange(half, device=t.device)  # shape: (half,)
            / half
        )
        # freqs shape: (half,)  例: (480,)

        # ---- Step 2: 時刻 × 周波数 の外積 ----
        # t[:, None]         shape: (B, 1)
        # freqs[None, :]     shape: (1, half)
        # 乗算 (broadcast)   shape: (B, half)
        # 【問題箇所 1】
        # ヒント: broadcast による outer product
        raise NotImplementedError(
            "TODO 1: SinusoidalTimeEmbedding.forward で args を計算せよ"
        )
        args = ...    # ← ここを書き換え. shape: (B, half)

        # ---- Step 3: sin と cos を横に連結 ----
        # torch.sin(args)     shape: (B, half)
        # torch.cos(args)     shape: (B, half)
        # cat(dim=-1)         shape: (B, hidden_dim = 2*half)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        # 最終 shape: (B, hidden_dim)  例: (B, 960)

### 8.2 Transformer ブロック

Flow Matching における速度場 $v_\theta$ の中身。VLM 出力を条件として参照しつつ、中間ノイズ状態 $x_t$ から action への方向を計算する。

### 中身: 標準 Transformer decoder の 1 層と同構造

各層で 3 段の処理を行い、これを $N$ 層 (教材版は 4 層、SmolVLA 論文は 16 層) 積み重ねます。

$$\begin{aligned}
h &\leftarrow \text{LayerNorm}(x) \\
x &\leftarrow x + \text{SelfAttn}(h) & \text{(action tokens 同士)} \\
h &\leftarrow \text{LayerNorm}(x) \\
x &\leftarrow x + \text{CrossAttn}(h,\ \text{VLM\_hidden}) & \text{(VLM 参照)} \\
h &\leftarrow \text{LayerNorm}(x) \\
x &\leftarrow x + \text{MLP}(h) & \text{(非線形変換)}
\end{aligned}$$

### 各段の役割

- **Self-attention (action tokens 内)**: chunk 内の各 step が互いに参照して時系列的な整合性を作る
- **Cross-attention (action → VLM 出力)**: action tokens が VLM の融合表現 (画像 + 指示 + robot state) から必要な情報を取り出す — **VLA の要**
- **MLP (position-wise feed-forward)**: attention だけでは足りない表現力を各位置で補う

### Attention 数式 (Vaswani et al. 2017)

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

- **Self-attention**: $Q = K = V = $ action tokens
- **Cross-attention**: $Q$ = action tokens, $K, V$ = VLM hidden state

### 実装ポイント: Q/K/V/O を明示的に分離

`nn.MultiheadAttention` は Q/K/V を単一の `in_proj_weight` にまとめてしまうため、
attention 内部の重みを個別に扱いにくくなります (可視化、診断、後から LoRA を貼るなどの実験)。
そこで **明示的な 4 つの `nn.Linear`** (`q_proj / k_proj / v_proj / o_proj`) で自前実装。
attention 演算そのものは PyTorch の `scaled_dot_product_attention` (SDPA) に任せ、
softmax + マスキング + Flash Attention を数値安定に一発でやってもらう。


In [ ]:
# ==============================================================
# 8.2 Transformer ブロック
# ==============================================================
#
# 【1 層の中身】(Pre-LN 型 Transformer)
#     x -> LN -> self_attn -> +残差 -> LN -> cross_attn(VLM 参照) -> +残差
#       -> LN -> MLP        -> +残差 -> 出力
#
# 【なぜ Q/K/V/O を分離するか】
#   nn.MultiheadAttention は in_proj_weight で Q/K/V を合体して持つため、
#   attention 内の個別重みを触りにくい (可視化 / 診断 / 後述の LoRA 追加実験 等が面倒).
#   個別 nn.Linear にしておくと重みを名前で参照できるので可読性が上がる.
#
# 【Attention の数式】(Vaswani et al. 2017)
#   Attention(Q, K, V) = softmax(Q K^T / sqrt(d_k)) V


class TransformerBlock(nn.Module):
    """Transformer decoder 1 層."""

    def __init__(self, hidden_dim: int, num_heads: int, mlp_ratio: int = 4):
        super().__init__()
        # multi-head の分割で余りが出ないよう割り切れる必要あり
        assert hidden_dim % num_heads == 0, "hidden_dim は num_heads の倍数にしてください"

        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        # 1 head が扱う次元 (例: hidden=960, heads=8 → head_dim=120)
        self.head_dim = hidden_dim // num_heads

        # ---- Self-attention の 4 つの Linear ----
        # bias=False は LLaMA/SmolVLM 系の慣例 (パラメタ節約 & 学習安定)
        self.self_q_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)  # (D → D)
        self.self_k_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)  # (D → D)
        self.self_v_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)  # (D → D)
        self.self_o_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)  # (D → D)

        # ---- Cross-attention の 4 つの Linear ----
        # Q は action tokens から、K/V は VLM hidden state から作る
        # (これが「画像+言語条件を action に注入する」ゲート)
        self.cross_q_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.cross_k_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.cross_v_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.cross_o_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)

        # ---- MLP (position-wise feed-forward, 4倍→GELU→戻す) ----
        # D → 4D → D
        self.mlp_in  = nn.Linear(hidden_dim, mlp_ratio * hidden_dim, bias=False)
        self.mlp_out = nn.Linear(mlp_ratio * hidden_dim, hidden_dim, bias=False)

        # ---- LayerNorm (Pre-LN 型) ----
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.norm3 = nn.LayerNorm(hidden_dim)

    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:
        # 入力  shape: (B, L, D)              -- D = num_heads * head_dim
        # 出力  shape: (B, num_heads, L, head_dim)
        #
        # multi-head attention は head 軸を独立させて並列計算するため、
        # チャンネル軸を (head, head_dim) に分割して並び替える。
        B, L, D = x.shape
        return x.view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        # after view:      (B, L, num_heads, head_dim)
        # after transpose: (B, num_heads, L, head_dim)

    def _merge_heads(self, x: torch.Tensor) -> torch.Tensor:
        # 入力  shape: (B, num_heads, L, head_dim)
        # 出力  shape: (B, L, D)               -- D = num_heads * head_dim
        B, H, L, Dh = x.shape
        return x.transpose(1, 2).contiguous().view(B, L, H * Dh)

    def _attention(
        self,
        q_proj: nn.Linear, k_proj: nn.Linear, v_proj: nn.Linear, o_proj: nn.Linear,
        q_input: torch.Tensor,   # (B, Lq, D)  ← Q の元
        k_input: torch.Tensor,   # (B, Lkv, D) ← K の元
        v_input: torch.Tensor,   # (B, Lkv, D) ← V の元
    ) -> torch.Tensor:           # (B, Lq, D)
        # Attention(Q, K, V) = softmax(Q K^T / sqrt(d_k)) V を計算.
        # 個々の射影と head 分割は自前で行い、softmax まわりだけ SDPA に任せる.

        # ---- Step 1: 射影 → head 分割 ----
        # 射影後:     (B, L, D)
        # 分割後:     (B, H, L, D_h)  (H = num_heads, D_h = head_dim)
        Q = self._split_heads(q_proj(q_input))   # (B, H, Lq,  D_h)
        K = self._split_heads(k_proj(k_input))   # (B, H, Lkv, D_h)
        V = self._split_heads(v_proj(v_input))   # (B, H, Lkv, D_h)

        # ---- Step 2: Scaled Dot-Product Attention ----
        # 数式: softmax(Q K^T / sqrt(D_h)) V
        # SDPA は数値安定 + Flash Attention v2 を自動採用する PyTorch 標準実装
        attn_out = F.scaled_dot_product_attention(Q, K, V)
        # attn_out shape: (B, H, Lq, D_h)

        # ---- Step 3: head 統合 → 出力射影 ----
        merged = self._merge_heads(attn_out)     # (B, Lq, D)
        return o_proj(merged)                    # (B, Lq, D)

    def forward(
        self,
        action_tokens: torch.Tensor,  # (B, chunk_size, D)   ← chunk 内の action 表現
        vlm_hidden:    torch.Tensor,  # (B, T_vlm,     D)   ← VLM 出力 (画像+text 融合)
    ) -> torch.Tensor:                # (B, chunk_size, D)
        # 以下すべて Pre-LN 型 (norm → sub-layer → 残差) で書く.
        # 残差接続によって層を深くしても勾配が消失しにくい (Deep Residual Learning, He et al. 2016).

        # ---- Sub-layer 1: Self-attention (chunk 内の相互作用) ----
        # action tokens が「自分自身」を Q/K/V にして相互参照する
        h = self.norm1(action_tokens)                                    # (B, chunk, D)
        action_tokens = action_tokens + self._attention(
            self.self_q_proj, self.self_k_proj,
            self.self_v_proj, self.self_o_proj,
            h, h, h,
        )                                                                 # (B, chunk, D)

        # ---- Sub-layer 2: Cross-attention (VLM 参照 = 条件付け) ----
        # ここで画像+言語の情報を吸う. Q は action, K/V は VLM.
        h = self.norm2(action_tokens)                                    # (B, chunk, D)
        action_tokens = action_tokens + self._attention(
            self.cross_q_proj, self.cross_k_proj,
            self.cross_v_proj, self.cross_o_proj,
            h,           # (B, chunk, D)  ← Q
            vlm_hidden,  # (B, T_vlm, D)  ← K
            vlm_hidden,  # (B, T_vlm, D)  ← V
        )                                                                 # (B, chunk, D)

        # ---- Sub-layer 3: Position-wise MLP ----
        # 各位置に独立に非線形変換をかける. (D → 4D → GELU → D)
        # GELU (Hendrycks & Gimpel 2016) は Transformer 系での定番活性化関数
        h = self.norm3(action_tokens)                                    # (B, chunk, D)
        mlp_h = F.gelu(self.mlp_in(h))                                   # (B, chunk, 4D)
        action_tokens = action_tokens + self.mlp_out(mlp_h)              # (B, chunk, D)

        return action_tokens                                             # (B, chunk, D)

### 8.3 Flow Matching Head

Transformer Block を $N$ 層積み上げた「velocity 予測モジュール」。以下 2 つの用途で使う:

- **学習時**: 中間状態 $x_t$ (ノイズと真 action の線形補間) を入力に、真の velocity $v = a - x_0$ を予測する回帰タスク
- **推論時**: pure noise から始めて、この Head が返す velocity に従って Euler 積分で action へ到達

### 入出力の shape 契約

| 引数 | shape | 意味 |
|---|---|---|
| `noisy_actions` | $(B, K, d_a)$ | 中間ノイズ状態 $x_t$ ($K$ = chunk_size, $d_a$ = action_dim) |
| `t` | $(B,)$ | 時刻 $t \in [0, 1]$ |
| `vlm_hidden` | $(B, L, D)$ | VLM の出力条件 |
| **戻り値** (velocity) | $(B, K, d_a)$ | $\frac{dx}{dt}$ の予測値 |

### 内部の流れ

1. **`noisy_actions` を $d_a \to D$ に射影** (`action_in`)  
2. **位置埋め込み + 時刻埋め込みを加算**  
3. **`num_layers` 段の Transformer Block を通す** (各層で VLM 参照)  
4. **$D \to d_a$ に戻す** (`action_out`) — 出力が velocity

```
入力  x_t (K,d_a) + t + vlm_hidden
   ↓ action_in       (d_a → D)
中間  hidden 表現 (K,D)  ← Transformer Block × num_layers
   ↓ action_out      (D → d_a)
出力  velocity (K,d_a)
```

### 実 SmolVLA との違い

構造的な違い (RoPE 位置埋め込み、self-attn 頻度、VLM 層数など) は **section 8.5 末尾の対照表** で一覧します。


In [ ]:
# ==============================================================
# 8.3 Flow Matching Head
# ==============================================================
#
# 【役割】 noise → action への「速度場 v_theta(x_t, t, cond)」を予測する Transformer.
#
# 【全体の shape 変換】
#     入力:     (B, K, d_a)              -- noisy actions
#     射影後:   (B, K, D)                -- action_in で d_a→D
#     +pos_emb: (B, K, D)                -- chunk 内位置埋め込み
#     +time_emb:(B, K, D)                -- 時刻を各 chunk 位置に broadcast
#     ↓ (Transformer Block を num_layers 段)
#     (B, K, D)                          -- VLM 参照しつつ精緻化
#     ↓ action_out で D→d_a
#     出力:     (B, K, d_a)              -- velocity


class FlowMatchingHead(nn.Module):
    """action chunk の velocity 場を条件付きで予測する Transformer."""

    def __init__(
        self,
        hidden_dim: int,     # D. VLM の hidden 次元と揃える (SmolVLM2 なら 960)
        action_dim: int,     # d_a. LIBERO は 7 (6-DoF + gripper)
        chunk_size: int,     # K. 未来何 step ぶんの action を一気に予測するか
        num_layers: int,     # Transformer の層数 (増やすと表現力↑ / 計算量↑)
        num_heads: int,      # multi-head の head 数 (D を割り切る必要)
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.action_dim = action_dim
        self.chunk_size = chunk_size

        # ---- action 次元 (d_a=7) ⇄ hidden 次元 (D=960) の入出力射影 ----
        # 入出力の解釈次元を橋渡しする部分. head 本体とは役割が違うので独立に扱う.
        self.action_in  = nn.Linear(action_dim, hidden_dim)  # (7 → 960)
        self.action_out = nn.Linear(hidden_dim, action_dim)  # (960 → 7)

        # ---- chunk 内の位置埋め込み (learnable parameter, zero-init) ----
        # Transformer は permutation invariant なので位置情報を明示的に注入する必要がある.
        # 実 SmolVLA では RoPE (Rotary Position Embedding, Su et al. 2021) を使うが、
        # 教材ではシンプルに learnable absolute embedding.
        # shape: (1, chunk_size, hidden_dim)  -- batch 軸は broadcast で対応
        self.pos_embed = nn.Parameter(torch.zeros(1, chunk_size, hidden_dim))

        # ---- 時刻 t 用の埋め込み器 (section 8.1) ----
        self.time_embed = SinusoidalTimeEmbedding(hidden_dim)

        # ---- Transformer 本体 (num_layers 段) ----
        # 各層の重みは blocks[i].self_q_proj など個別 Linear で参照可能.
        self.blocks = nn.ModuleList([
            TransformerBlock(hidden_dim, num_heads)
            for _ in range(num_layers)
        ])

    def forward(
        self,
        noisy_actions: torch.Tensor,  # (B, K,     d_a)   -- 中間ノイズ状態 x_t
        t:             torch.Tensor,  # (B,)              -- 時刻 t ∈ [0, 1]
        vlm_hidden:    torch.Tensor,  # (B, T_vlm, D)     -- VLM 出力 (条件)
    ) -> torch.Tensor:                # (B, K,     d_a)   -- velocity 予測
        # ---- Step 1: action 次元 → hidden 次元へ射影 ----
        # action_in(noisy_actions):  (B, K, d_a) → (B, K, D)
        x = self.action_in(noisy_actions) + self.pos_embed
        # x shape: (B, K, D)
        # pos_embed shape: (1, K, D) が batch 軸に broadcast されて加算される

        # ---- Step 2: 時刻埋め込みを chunk 全 step にブロードキャスト加算 ----
        # time_embed(t):        (B, D)
        # time_embed(t)[:, None, :]:  (B, 1, D)  ← chunk 軸に 1 を挟む
        # x に broadcast 加算   → (B, K, D)
        x = x + self.time_embed(t)[:, None, :]

        # ---- Step 3: Transformer 層で VLM 情報を吸わせる ----
        # 各層で self-attn + cross-attn(VLM 参照) + MLP を通る.
        # shape は各層内で保存される: (B, K, D)
        for block in self.blocks:
            x = block(x, vlm_hidden)  # (B, K, D)

        # ---- Step 4: hidden 次元 → action 次元へ戻して velocity として出力 ----
        # action_out:  (B, K, D) → (B, K, d_a)
        return self.action_out(x)

### 8.4 SmolVLM2 をロードして完全に凍結する

Hugging Face から `HuggingFaceTB/SmolVLM2-500M-Video-Instruct` (500M パラメタ) をロード。

**ポイント**:
- 精度は **bf16** (RTX 40 系) または **fp16** (それ以外) で読み込む → メモリ半分
- `eval()` + `requires_grad_(False)` で **完全に凍結** (勾配を流さない)
- `RealVLMEncoder` は「画像 (PIL/numpy) + タスク文 → hidden state」の便利ラッパ

**初回のみ 500M の重みを DL する**ので数分かかります (2 回目以降は HF cache から)。

In [ ]:
# ==============================================================
# 8.4 SmolVLM2-500M-Video-Instruct をロード + 完全 freeze
# ==============================================================
from transformers import AutoModelForImageTextToText, AutoProcessor
from PIL import Image
import numpy as np

VLM_MODEL_ID = "HuggingFaceTB/SmolVLM2-500M-Video-Instruct"

# bf16 対応 GPU (A100/H100/L40/RTX40 系) なら bf16、それ以外は fp16
# fp32 だと 500M × 4byte = 2GB とメモリを食うので現実的でない
_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print(f"Loading VLM ({VLM_MODEL_ID}) in {_dtype}...", flush=True)

# processor: 画像リサイズ・正規化・トークン化を全部担当する統合オブジェクト
vlm_processor = AutoProcessor.from_pretrained(VLM_MODEL_ID)

# 本体
vlm_model = AutoModelForImageTextToText.from_pretrained(
    VLM_MODEL_ID,
    torch_dtype=_dtype,
).to("cuda")

# ---- 完全 freeze (これが本課題の縛り: VLM は学習対象外) ----
vlm_model.eval()
for p in vlm_model.parameters():
    p.requires_grad = False

# hidden_dim = 960 (以降の action head の入出力次元)
VLM_HIDDEN_DIM = vlm_model.config.text_config.hidden_size
print(f"VLM loaded. hidden_dim = {VLM_HIDDEN_DIM}")

# 学習可能な params が 0 個であることを確認
_train = sum(p.numel() for p in vlm_model.parameters() if p.requires_grad)
_total = sum(p.numel() for p in vlm_model.parameters())
print(f"VLM params: total {_total/1e6:.1f}M, trainable {_train/1e6:.1f}M  ← 0 のはず")


class RealVLMEncoder(nn.Module):
    """SmolVLM2 をラップ: (画像 list, task 文 list) → hidden state.

    - VLM は凍結済みなので forward は torch.no_grad
    - 内部で processor が chat template を組み立て、pixel_values / input_ids に変換
    - 最後の layer の hidden_states (画像 tokens + text tokens の融合表現) を返す
    """

    def __init__(self, model, processor, hidden_dim: int, dtype):
        super().__init__()
        self.model = model
        self.processor = processor
        self.hidden_dim = hidden_dim
        self._dtype = dtype

    @torch.no_grad()
    def forward(self, images, task_texts: list[str]) -> torch.Tensor:
        # images: PIL / numpy / torch tensor どれでも受け付ける (バッチは list で渡す)
        # task_texts: 各サンプルのタスク説明文 (長さ B の list)

        # 入力を PIL に統一 (processor は PIL を期待する)
        pil_images = []
        for img in images:
            if isinstance(img, np.ndarray):
                pil_images.append(Image.fromarray(img.astype(np.uint8)))
            elif isinstance(img, torch.Tensor):
                arr = img.cpu().numpy()
                if arr.ndim == 3 and arr.shape[0] == 3:  # (C,H,W) → (H,W,C)
                    arr = arr.transpose(1, 2, 0)
                pil_images.append(Image.fromarray(arr.astype(np.uint8)))
            else:
                pil_images.append(img)

        # SmolVLM2 の chat template で「画像 + 指示文」の prompt に組み立てる
        prompts = []
        for txt in task_texts:
            messages = [{
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": txt},
                ],
            }]
            prompts.append(
                self.processor.apply_chat_template(
                    messages, add_generation_prompt=True
                )
            )

        # tokenize + preprocess. padding=True で可変長を揃える
        inputs = self.processor(
            text=prompts,
            images=pil_images,
            return_tensors="pt",
            padding=True,
        )

        # モデルの device / dtype に揃える
        device = next(self.model.parameters()).device
        inputs = {
            k: (v.to(device=device, dtype=self._dtype)
                if v.dtype in (torch.float16, torch.float32, torch.bfloat16)
                else v.to(device))
            for k, v in inputs.items()
        }

        # 最後の hidden state を取りたいので output_hidden_states=True
        outputs = self.model(**inputs, output_hidden_states=True)

        # hidden_states は各 layer 分の tuple、[-1] が最終 layer
        # shape: (B, L, hidden_dim). L は画像 tokens + テキスト tokens の合計長.
        return outputs.hidden_states[-1]


# 動作確認: 適当な画像とタスク文でエンコード
vlm_encoder = RealVLMEncoder(vlm_model, vlm_processor, VLM_HIDDEN_DIM, _dtype)

_dummy_img = [Image.new("RGB", (256, 256), color=(128, 64, 200))]
_dummy_task = ["pick up the black bowl on the plate"]
_hidden = vlm_encoder(_dummy_img, _dummy_task)
print(f"VLM output shape: {tuple(_hidden.shape)}  dtype: {_hidden.dtype}")
# 期待: (1, L, 960). L はプロンプト長 + 画像トークン数の合計

### 8.5 VLA を組み立てる

Vision-Language 部分 (`RealVLMEncoder`, frozen) と Action 部分 (`FlowMatchingHead`, trainable) を結合。

### 学習用: `compute_loss`

Flow Matching loss を計算する:

$$\mathcal{L}(\theta) = \big\| v_\theta(x_t, t, c) - (a - x_0) \big\|^2$$

- $a$ = 真の action chunk (batch 内でサンプルされる)
- $x_0 \sim \mathcal{N}(0, I)$ = ガウシアンノイズ
- $t \sim U(0, 1)$ = 一様分布からの時刻
- $x_t = (1-t) x_0 + t a$ = 線形補間
- $c$ = VLM 出力 (射影後、画像・言語条件)
- $v_\theta$ = FlowMatchingHead

### 推論用: `sample_actions`

Euler 法で ODE を数値積分:

$$x_{t + \Delta t} = x_t + v_\theta(x_t, t, c) \cdot \Delta t, \quad \Delta t = 1 / N$$

$x_0 \sim \mathcal{N}(0, I)$ から始めて $t = 0 \to 1$ まで $N$ ステップ進めれば $x_1 \approx a$。
本教材では $N = 10$ (SmolVLA 公式のデフォルトと同じ) を使用。必要なステップ数はタスク / solver / 学習状況に依存する。

---

## 📊 実 SmolVLA (LeRobot 公式) と教材版の違い

> **⚠️ 本教材は公式実装の再実装ではなく、意図的に簡略化した「教育用簡略実装」です**。
> VLA の主要概念 (VLM 凍結 + Flow Matching + action chunking) を PyTorch で書き下すことに集中しており、
> 内部の attention 構造・時刻埋め込みの数式・KV cache 等は公式版と細かい差があります。

実 SmolVLA (`lerobot/smolvla_libero_plus`) は大規模事前学習を経た本格モデル。
教材版はそれを **from-scratch × 50 episode で回せる規模** まで削ぎ落としたものです。
以下、違いを 5 グループに整理して並べます。

### ① 規模 (パラメタ・サイズ)

- **`chunk_size`** — 実 SmolVLA: 50 / 教材版: **8**
  - 一度に予測する未来の action 数。50 は 2.5 秒 (@20Hz)、8 は 0.4 秒。長いほど「先読み」が効くが学習/推論とも重い。
- **`num_transformer_layers`** — 実 SmolVLA: 16 / 教材版: **4**
  - Transformer の層数。深いほど表現力↑ / 計算量↑ / 過学習リスク↑。
- **`transformer_hidden_dim`** — 実 SmolVLA: 720 (= 960 × 0.75) / 教材版: **256**
  - Transformer 内部の hidden 次元。実 SmolVLA も VLM (960) より小さく (0.75×) している。教材版はさらに 256 まで削減。
- **`num_heads`** — 実 SmolVLA: 15 (= 720/48) / 教材版: **8** (= 256/32)
  - Multi-head attention の head 数。
- **学習対象 params** — 実 SmolVLA: 数十〜数百 M / 教材版: 約 4.5 M
  - 教材版は上記削減で 96% 減。

### ② アーキテクチャの細かい違い

- **VLM の使い方**
  - 実 SmolVLA: 32 層のうち先頭 16 層のみ通す (`num_vlm_layers=16`)
  - 教材版: 32 層全部通した最終 hidden を使う
  - 影響: 教材版のほうが情報量↑だが計算量も↑
- **位置埋め込み**
  - 実 SmolVLA: Rotary Position Embedding (RoPE) — 各 attention 内で回転行列として注入
  - 教材版: learnable 絶対 embedding — `nn.Parameter` 1 個を加算するだけ
  - 影響: RoPE の方が chunk_size を可変にしやすく汎化性能も一般に高い
- **self-attn の頻度**
  - 実 SmolVLA: `self_attn_every_n_layers=2` → 2 層ごとに 1 回だけ self-attn (残りは cross-attn のみ)
  - 教材版: 各層で常に self+cross+MLP の 3 段
  - 影響: 実 SmolVLA の方が計算量を抑えつつ長い依存を扱える
- **Attention 全体構造**
  - 実 SmolVLA: GQA + RoPE + interleaved self/cross + custom mask
  - 教材版: 通常 MHA + learnable 絶対位置埋め込み + 毎層 self+cross
  - 影響: 構造として大きく異なる (Q/K/V/O を個別 Linear で持つ点のみ共通)
- **Multi-view**
  - 実 SmolVLA: 複数 view 対応 (view 数はデータセット / checkpoint 依存)
  - 教材版: front カメラ 1 個のみ
  - 影響: 教材版は情報量が減るぶん収束は簡単

### ③ 入力前処理

- **画像サイズ**
  - 実 SmolVLA: 512×512 (padding 付きリサイズ)
  - 教材版: 256×256 (LIBERO env のネイティブ)
  - 影響: 教材版の方が VRAM 節約
- **タスク文の長さ**
  - 実 SmolVLA: 最大 48 token に truncate
  - 教材版: 制限なし (processor 任せ)
  - 影響: LIBERO のタスク文は短いので実質差なし
- **State ベクトル**
  - 実 SmolVLA: state projection で 1 token として結合
  - 教材版: 同じく state_proj で 1 token を追加
  - 影響: 教材版もサポート (`state_dim=8` = LIBERO の joints+gripper)

### ④ 学習ハイパラ

- **Warmup steps** — 実 SmolVLA `train_config` デフォルト: 1,000 / 教材版: 30
  - 教材版は step 数が少ないので比例縮小
- **Total steps** — 実 SmolVLA: 30,000 / 教材版: 300 (`TRAIN_STEPS_PY`)
  - 教材はデモ用
- **Peak learning rate** — 実 SmolVLA: 1e-4 / 教材版: 3e-4
  - 教材は from-scratch なのでやや高め
- **Decay final lr** — 実 SmolVLA: 2.5e-6 / 教材版: peak × 0.1 (= 3e-5)
  - 単純な cosine decay に簡略化
- **Weight decay** — 実 SmolVLA: 1e-10 / 教材版: 1e-4
  - 教材の方が強い regularization
- **Gradient clip norm** — 実 SmolVLA: 10 / 教材版: 1.0
  - 教材の方が保守的
- **Optimizer betas** — 実 SmolVLA: (0.9, 0.95) / 教材版: (0.9, 0.999) default
  - 教材は torch デフォルトのまま

### ⑤ 事前学習の有無 (最重要の違い)

- **Action head 初期値**
  - 実 SmolVLA: Community Datasets (150k エピソード超) で事前学習済み
  - 教材版: 完全にランダム初期化
- **LoRA**
  - 実 SmolVLA: 事前学習済み action head を保ったまま、LIBERO-Spatial に LoRA で追従
  - 教材版: 使わない (random に LoRA を貼っても利点なし。詳細は section 8.6)
- **期待成功率 (LIBERO-Spatial)**
  - 実 SmolVLA: 70〜90%
  - 教材版: ほぼ 0% (失敗前提) — 小データ + 小モデル + 300 step では実用的な成功率は出ない

→ 教材版はあくまで「PyTorch で VLA を実装 → 学習 → 評価が回るところまで」を体験することが目的。
実 SmolVLA との性能勝負ではありません。数値スコアではなく **「動く実装を書けたか」** が評価対象 (Basic 課題)。

### なぜ小さくしたか (再掲)

- 事前学習済み重みが無い **from-scratch**、しかも 50 episode の小データ
- 論文サイズの Transformer (16層×720dim) は、この設定では **過学習 & 収束遅い**
- Colab の GPU でも現実的な時間で回るサイズに合わせた

### VLM 次元とのブリッジ

`transformer_hidden_dim` を VLM の hidden (960) より小さくすると、
VLM 出力を Transformer 側で使うために **線形射影 (`vlm_projector`)** が必要:

$$h_{\text{tr}} = W_{\text{proj}} \cdot \text{VLM\_hidden}, \quad W_{\text{proj}} \in \mathbb{R}^{256 \times 960}$$

実 SmolVLA でも `expert_width_multiplier = 0.75` で VLM (960) → Transformer (720) の縮小をしており、教材版はそれをさらに押し進めた形 (`expert_width_multiplier` は LeRobot 側の config 名なのでそのまま)。

### 改造案 (Advanced 向け)

- `chunk_size` を 16, 32 に上げる (先読みを長くする)
- `transformer_hidden_dim` を 512 に増やす (表現力↑)
- `num_transformer_layers` を 8 に増やす (深さ↑)
- 位置埋め込みを RoPE に置き換える (実 SmolVLA と揃える)
- Multi-view (front + wrist) 対応にする — `RealVLMEncoder.forward` を 2 画像対応に
- Data augmentation を加える (画像 crop / colour jitter 等)


In [ ]:
# ==============================================================
# 8.5 SmolVLA 統合モジュール
# ==============================================================
#
# 【役割】 Vision-Language (frozen SmolVLM2) + State + Action (自作 FlowMatchingHead) を結合.
#
# 【VLA の "S"】 SmolVLA は画像・言語に加えて **現在のロボット状態 (state)** も入力に取る.
#   典型的には joint positions (7) + gripper qpos (1) = 8 次元. LIBERO データセットの
#   observation.state と一致する. state を Linear で transformer_hidden_dim に射影し、
#   VLM 出力の末尾に 1 つの state token として連結する.
#   (概念的には SmolVLA 公式と同じ発想だが、公式では VLM 内部処理と統合される
#    点が異なる. 詳細は section 8.5 の対照表を参照.)
#
# 【学習用 compute_loss の数式】
#     L(theta) = || v_theta(x_t, t, cond) - (a - x_0) ||^2
#     where  x_0 ~ N(0, I),  t ~ U(0, 1),  x_t = (1-t) x_0 + t a,
#            cond = [vlm_projector(VLM(images, task_text)),  state_proj(state)]
#
# 【推論用 sample_actions の数式】(Euler 積分, t: 0 -> 1)
#     x <- sample from N(0, I)                          [初期化]
#     for step in 0..N-1:
#         t = step / N
#         x <- x + v_theta(x, t, cond) * (1/N)          [Euler 1 歩]
#     return x   # ≈ 真の action


class SmolVLA(nn.Module):
    """SmolVLA 全体. VLM は frozen、Action head は trainable、state は projection 経由."""

    def __init__(
        self,
        vlm_encoder: "RealVLMEncoder",   # 凍結済みラッパ (section 8.4)
        action_dim: int = 7,             # LIBERO: 6-DoF + gripper = 7 次元
        state_dim: int = 8,              # LIBERO の observation.state (joint 7 + gripper 1)
        chunk_size: int = 8,             # 予測する連続 action 数 (実 SmolVLA は 50)
        num_transformer_layers: int = 4, # Transformer の層数
        num_heads: int = 8,              # multi-head の head 数
        transformer_hidden_dim: int | None = None,  # None なら VLM の hidden dim (960) をそのまま使う
    ):
        super().__init__()
        self.vlm_encoder = vlm_encoder

        # VLM の元 hidden (SmolVLM2 なら 960)
        vlm_hidden = vlm_encoder.hidden_dim

        # Transformer 内部の hidden を独立に指定できるようにする
        if transformer_hidden_dim is None:
            transformer_hidden_dim = vlm_hidden
        self.transformer_hidden_dim = transformer_hidden_dim

        # VLM 出力 (dim=960) を Transformer 用の次元 (例 256) に線形射影
        if transformer_hidden_dim != vlm_hidden:
            self.vlm_projector = nn.Linear(vlm_hidden, transformer_hidden_dim, bias=False)
        else:
            self.vlm_projector = nn.Identity()

        # ★ state を transformer_hidden_dim に射影する層 (VLA の "S" 部分)
        # 8 次元の robot state → 256 次元の埋め込みトークン 1 個
        self.state_proj = nn.Linear(state_dim, transformer_hidden_dim)

        # 後方互換のため hidden_dim も expose
        self.hidden_dim = transformer_hidden_dim
        self.action_dim = action_dim
        self.state_dim = state_dim
        self.chunk_size = chunk_size

        # Flow Matching Head は Transformer の hidden 次元で構築
        self.flow_head = FlowMatchingHead(
            hidden_dim=transformer_hidden_dim,
            action_dim=action_dim,
            chunk_size=chunk_size,
            num_layers=num_transformer_layers,
            num_heads=num_heads,
        )

    def _encode_cond(
        self, images, task_texts, states: torch.Tensor
    ) -> torch.Tensor:
        # 1) VLM から画像+言語の融合表現 (勾配なし)
        hidden = self.vlm_encoder(images, task_texts)   # (B, L, 960)
        # 2) dtype を action head に合わせる (VLM: fp16/bf16, head: fp32)
        target_dtype = next(self.flow_head.parameters()).dtype
        hidden = hidden.to(dtype=target_dtype)
        # 3) Transformer の hidden 次元に射影 (Identity or Linear)
        cond_vlm = self.vlm_projector(hidden)            # (B, L, transformer_hidden_dim)
        # 4) state を射影して 1 トークンとして末尾に連結
        state_token = self.state_proj(
            states.to(dtype=cond_vlm.dtype, device=cond_vlm.device)
        )                                                # (B, transformer_hidden_dim)
        state_token = state_token.unsqueeze(1)           # (B, 1, transformer_hidden_dim)
        # 5) [VLM トークン列 | state トークン 1 個] を connect
        return torch.cat([cond_vlm, state_token], dim=1)  # (B, L+1, transformer_hidden_dim)

    def compute_loss(
        self,
        images,                         # 長さ B の list (PIL/np/tensor 何でも可)
        task_texts: list[str],          # 長さ B の list (各サンプルの task 説明文)
        states: torch.Tensor,           # (B, state_dim) 現在のロボット状態
        actions: torch.Tensor,          # (B, K, d_a)  正規化済み真 action
    ) -> torch.Tensor:                  # scalar (MSE loss)
        """Flow Matching 学習損失.

        アルゴリズム (Lipman et al. 2023, Conditional Flow Matching):
          1. 各サンプルにノイズ x_0 ~ N(0, I) を用意                     (B, K, d_a)
          2. 各サンプルに時刻 t ~ Uniform(0, 1) を用意                   (B,)
          3. 線形パス上の中間点 x_t = (1-t) x_0 + t a を計算            (B, K, d_a)
          4. 真の速度 v* = a - x_0 (直線パスなので path 上で定数)      (B, K, d_a)
          5. VLM+state の融合条件 cond を取得                            (B, L+1, D)
          6. ネットワークに (x_t, t, cond) を渡して速度を予測 v_pred    (B, K, d_a)
          7. MSE(v_pred, v*) を返す                                      scalar
        """
        B = actions.size(0)
        device = actions.device

        # ---- 条件 (VLM + state, 画像/言語/状態の融合) ----
        vlm_hidden = self._encode_cond(images, task_texts, states).to(device)
        # vlm_hidden shape: (B, L+1, transformer_hidden_dim)

        # ---- Step 1: ノイズと時刻をサンプル ----
        noise = torch.randn_like(actions)     # (B, K, d_a)  x_0 ~ N(0, I)
        t = torch.rand(B, device=device)      # (B,)         t ~ U(0, 1)

        # ---- Step 2: 線形補間で中間点 x_t を作る ----
        tv = t[:, None, None]                 # (B, 1, 1) — chunk / action 軸に broadcast
        # 【問題箇所 2】
        # 数式: x_t = (1 - t) * x_0 + t * a       (tv=t, noise=x_0, actions=a)
        # 出力 shape: (B, K, d_a)
        raise NotImplementedError(
            "TODO 2: 線形補間 x_t を実装せよ (SmolVLA.compute_loss)"
        )
        xt = ...    # ← ここを書き換え

        # ---- Step 3: 真の速度 = a - x_0 (直線パスなので定数場) ----
        # 【問題箇所 3】
        # 数式: v* = a - x_0     (path 上で定数)
        # 出力 shape: (B, K, d_a)
        raise NotImplementedError(
            "TODO 3: 真の速度 target_v を計算せよ (SmolVLA.compute_loss)"
        )
        target_v = ...    # ← ここを書き換え

        # ---- Step 4: ネットワークで速度を予測 ----
        pred_v = self.flow_head(xt, t, vlm_hidden)  # (B, K, d_a)

        # ---- Step 5: MSE loss ----
        return F.mse_loss(pred_v, target_v)

    @torch.no_grad()
    def sample_actions(
        self,
        images,
        task_texts: list[str],
        states: torch.Tensor,             # (B, state_dim) 現在のロボット状態
        num_steps: int = 10,
    ) -> torch.Tensor:                 # (B, K, d_a)  正規化空間の action chunk
        """Euler 法で ODE を数値積分して action chunk を復元."""
        device = next(self.flow_head.parameters()).device

        # VLM+state 条件 (勾配なし, 事前計算して使い回す)
        vlm_hidden = self._encode_cond(images, task_texts, states).to(device)
        # vlm_hidden shape: (B, L+1, transformer_hidden_dim)

        B = vlm_hidden.size(0)

        # ---- 初期化: pure noise ----
        x = torch.randn(B, self.chunk_size, self.action_dim, device=device)
        # x shape: (B, K, d_a)

        dt = 1.0 / num_steps  # 例: N=10 なら dt = 0.1

        # ---- Euler 積分ループ ----
        for step in range(num_steps):
            t_now = torch.full((B,), step * dt, device=device)  # (B,)
            v = self.flow_head(x, t_now, vlm_hidden)             # (B, K, d_a)
            # 【問題箇所 4】
            # 数式: x_{t+dt} = x_t + v(x, t, cond) * dt
            # 出力 shape: (B, K, d_a)
            raise NotImplementedError(
                "TODO 4: Euler の 1 歩を実装せよ (SmolVLA.sample_actions)"
            )
            x = ...    # ← ここを書き換え

        # 最終 x は「正規化された action 空間」の値.
        # 呼び出し側で denormalize_actions() して実 action に戻す.
        return x


# ==============================================================
# インスタンス化: 論文サイズ (コメントアウト) と 教材サイズ を対比
# ==============================================================

# ---- 論文相当サイズ (SmolVLA / LeRobot 公式デフォルト、参考) ----
#
# smolvla = SmolVLA(
#     vlm_encoder=vlm_encoder,
#     action_dim=7,
#     state_dim=8,
#     chunk_size=50,                # SmolVLA original: 50 step (2.5 秒 @ 20Hz)
#     num_transformer_layers=16,    # SmolVLA original: num_vlm_layers と同じ
#     num_heads=15,                 # SmolVLA original: hidden 720 / head_dim 48
#     transformer_hidden_dim=720,   # VLM_hidden × 0.75 (公式 expert_width_multiplier=0.75)
# ).to("cuda")

# ---- 教材用サイズ (from-scratch × 50 episode 向けに小型化) ----
smolvla = SmolVLA(
    vlm_encoder=vlm_encoder,
    action_dim=7,
    state_dim=8,                    # LIBERO の observation.state 次元
    chunk_size=8,                   # 8 step 予測 (0.4 秒). 学習も推論も軽く
    num_transformer_layers=4,       # 16 → 4 に圧縮
    num_heads=8,                    # transformer_hidden_dim=256 に合わせて (256/32 = 8)
    transformer_hidden_dim=256,     # 720 → 256 に圧縮 (params が数分の 1 に)
).to("cuda")

# ---- 学習可能 params と凍結 params の内訳 ----
_train = sum(p.numel() for p in smolvla.parameters() if p.requires_grad)
_frozen = sum(p.numel() for p in smolvla.parameters() if not p.requires_grad)
print(f"SmolVLA — trainable: {_train/1e6:.2f}M  frozen: {_frozen/1e6:.2f}M")
print(f"  (frozen が SmolVLM2-500M ぶん、trainable が Action head + projector + state_proj)")
print(f"  transformer_hidden_dim = {smolvla.transformer_hidden_dim}, "
      f"chunk_size = {smolvla.chunk_size}, "
      f"state_dim = {smolvla.state_dim}, "
      f"num_transformer_layers = {len(smolvla.flow_head.blocks)}")

### 8.6 学習対象パラメタの確認

VLM (500M) は凍結、Action head (数M) が全部学習対象、という構成になっているか **数字で確認** します。

### なぜここで LoRA を使わないか

LoRA (Hu et al., ICLR 2022) は本来:

$$W = W_0 + \frac{\alpha}{r} B A, \quad W_0\ \text{は事前学習済みで凍結、}\ BA\ \text{だけ学習}$$

という「**事前学習済み重み $W_0$ を保ったまま少量の差分 $BA$ だけ学習**」の手法。
しかし今回の Action head は **from-scratch** (random init) なので $W_0$ に「守るべき知識」がなく、
LoRA の恩恵はほとんどありません (単に低ランク制約が掛かった学習になるだけ)。

**LoRA が意味を持つのは事前学習済みモデルを追加学習する場面**。 それは Advanced の
LeRobot 版で扱います (`lerobot/smolvla_libero_plus` の action head を warm-start し、
LoRA で LIBERO-Spatial に追従させる、という教科書通りのやり方)。

ここでは素朴に Action head 全体をフル学習します。

In [ ]:
# ==============================================================
# 8.6 学習対象パラメタの内訳を確認する
# ==============================================================
# 期待される内訳:
#   frozen    ≈ 500 M   ← SmolVLM2-500M (画像+言語部分、丸ごと凍結)
#   trainable ≈ 数 M    ← FlowMatchingHead (自作 action head、フル学習)


def summarize_params(model: nn.Module, top_k: int = 8) -> None:
    """学習可能 / 凍結パラメタの合計と、トップ K モジュールの内訳を表示."""
    total_trainable = 0
    total_frozen = 0
    per_module_trainable: dict[str, int] = {}

    for name, p in model.named_parameters():
        n = p.numel()
        if p.requires_grad:
            total_trainable += n
            # モジュール単位 (先頭 2 階層) で集約
            key = ".".join(name.split(".")[:2])
            per_module_trainable[key] = per_module_trainable.get(key, 0) + n
        else:
            total_frozen += n

    total = total_trainable + total_frozen
    pct = 100.0 * total_trainable / max(total, 1)
    print(f"Total params    : {total/1e6:8.2f} M")
    print(f"  Frozen (VLM)  : {total_frozen/1e6:8.2f} M")
    print(f"  Trainable     : {total_trainable/1e6:8.2f} M  ({pct:.2f}% of total)")
    print()

    # 学習対象モジュールを大きい順に top_k 個表示
    print(f"Trainable module breakdown (top {top_k}):")
    ranked = sorted(per_module_trainable.items(), key=lambda kv: -kv[1])
    for name, n in ranked[:top_k]:
        print(f"  {name:40s} {n/1e6:8.3f} M")


summarize_params(smolvla)

# 全学習パラメタが flow_head 配下にあることを確認 (VLM 側は 0)
_vlm_train = sum(
    p.numel() for p in smolvla.vlm_encoder.parameters() if p.requires_grad
)
_head_train = sum(
    p.numel() for p in smolvla.flow_head.parameters() if p.requires_grad
)
print()
print(f"vlm_encoder trainable: {_vlm_train/1e6:.3f} M  ← 0 のはず")
print(f"flow_head   trainable: {_head_train/1e6:.3f} M  ← ここが学習される")

## 9. 学習

section 8 で組み立てた `smolvla` を、LIBERO Spatial の 50 エピソード (section 7 で選抜) で学習させる。

### やること
1. `LeRobotDataset` から LIBERO の観測画像 / タスク文 / 未来 action chunk を取り出す
2. 全 action の平均・分散で正規化 (これをやると収束が速い)
3. AdamW + warmup+cosine 学習率で PyTorch loop を回す

### 9.1 LIBERO データセットを読み込む

LeRobot の `LeRobotDataset` にラップしてもらえば、以下が 1 行で揃う:
- 現在の観測画像 (front カメラ)
- 現在のタスク説明文
- 未来 `chunk_size` step ぶんの action

**`delta_timestamps`** で「相対時刻オフセット」を指定すると、
未来のフレーム/action を並べて取ってくれる。今回は action だけ chunk_size 個欲しいので action だけ指定。

In [ ]:
# ==============================================================
# 9.1 LIBERO データセット (chunk_size ぶんの future action 付き)
# ==============================================================
from lerobot.datasets.lerobot_dataset import LeRobotDataset

# smolvla の chunk_size と揃える (揃わないと loss 計算で shape mismatch)
CHUNK_SIZE_TRAIN = smolvla.chunk_size  # 8

# LIBERO の記録レート = 20 Hz. delta_timestamps は「秒単位のオフセット」で指定するので
# i-th 未来 step は i / fps 秒後.
_DATASET_FPS = 20.0
delta_timestamps = {
    "action": [i / _DATASET_FPS for i in range(CHUNK_SIZE_TRAIN)],
}

print(f"Loading LIBERO dataset with {len(EPISODE_INDICES)} episodes...", flush=True)

# dataset の初回ロードは HF から episode ぶんの動画をひっぱるので時間がかかる.
# その進捗を見せるため、このスコープだけ progress bar を有効化.
from huggingface_hub.utils import enable_progress_bars, disable_progress_bars
enable_progress_bars()
try:
    # section 7 で選抜した 50 エピソードだけ使う
    train_dataset = LeRobotDataset(
        DATASET_REPO,
        revision=DATASET_REVISION,
        episodes=EPISODE_INDICES,
        delta_timestamps=delta_timestamps,
    )
finally:
    disable_progress_bars()

# 1 サンプルの中身を確認
_sample = train_dataset[0]
print("\n=== sample keys ===")
for k, v in _sample.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k:40s} shape={tuple(v.shape)}  dtype={v.dtype}")
    else:
        print(f"  {k:40s} type={type(v).__name__}  val={str(v)[:60]}")

print(f"\n# training samples total: {len(train_dataset)}")

# --- state 定義の可視化 (rollout 時と一致するか要確認) ---
_state0 = _sample["observation.state"]
print(f"\n=== observation.state の実サンプル ===")
print(f"  shape:  {tuple(_state0.shape)}")
print(f"  values: {_state0.numpy().round(3)}")
print("↑ 本教材では LIBERO env の joints.pos (7) + gripper.qpos[0] (1) を")
print("   同じ順序で 8 次元 state として構成する (section 10.1 参照).")

### 9.2 action の mean / std を計算 (正規化)

LIBERO の生 action は次元ごとに range がバラバラ:
- 位置 (x, y, z, roll, pitch, yaw) は ±0.1 前後
- gripper 開閉は {-1, +1}

そのまま学習すると次元間で loss の寄与が偏る。全 action の mean/std で正規化してから学習し、
推論時は逆変換で戻す。実 SmolVLA も同じことを processor 内でやっている。

In [ ]:
# ==============================================================
# 9.2 action の平均・分散で正規化するヘルパを作る
# ==============================================================
# 50 エピソード全部の action を集めて (chunk_size × N, action_dim) に flatten し、
# 次元ごとに mean / std を計算する.

print("Collecting action statistics...", flush=True)
_all_actions = []
for i in range(len(train_dataset)):
    _all_actions.append(train_dataset[i]["action"])
_all_actions = torch.stack(_all_actions).view(-1, 7)   # (N, 7)

# std が 0 だと割り算で NaN が出るので下限を設ける
action_mean = _all_actions.mean(dim=0)                 # (7,)
action_std = _all_actions.std(dim=0).clamp_min(1e-3)   # (7,)

# GPU 上に置く (学習中の毎 step の正規化に使うため)
action_mean = action_mean.to("cuda", dtype=torch.float32)
action_std = action_std.to("cuda", dtype=torch.float32)

print(f"action_mean: {action_mean.cpu().numpy().round(3)}")
print(f"action_std : {action_std.cpu().numpy().round(3)}")


def normalize_actions(a: torch.Tensor) -> torch.Tensor:
    # 学習前: 真 action - mean を std で割ると 平均 0 / 分散 1 に近づく
    return (a - action_mean) / action_std


def denormalize_actions(a: torch.Tensor) -> torch.Tensor:
    # 推論後: モデル出力に std を掛けて mean を足すと元スケールへ
    return a * action_std + action_mean

### 9.3 PyTorch 学習ループ

素の PyTorch で書き下ろした学習ループ:

- **AdamW** で学習可能な param のみ更新
- **linear warmup → cosine decay** の学習率スケジュール
- **grad clip** で勾配爆発対策
- 100 step ごとに loss / lr を print
- 最終 step で checkpoint を保存

デフォルトは **300 step のデモ** (数分)。 本気で試したければ `TRAIN_STEPS_PY = 3000` に。

In [ ]:
# ==============================================================
# 9.3 PyTorch 学習ループ
# ==============================================================
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
import time

# ---- PyTorch 版の学習ハイパラ ----
# ここで定義する TRAIN_STEPS_PY 等が PyTorch 自作版 (このセル) を制御する.
TRAIN_STEPS_PY = 300           # 学習ステップ数 (デモ用に控えめ. 本気なら 3000)
TRAIN_BATCH_SIZE_PY = 1        # VLM が重いので batch=1 が無難
TRAIN_LR_PY = 3e-4             # peak lr
TRAIN_WARMUP_PY = 30           # linear warmup 区間
TRAIN_LOG_FREQ_PY = 20         # loss を print する頻度
TRAIN_NUM_WORKERS_PY = 0       # Colab 環境では 0 が安全

PY_CHECKPOINT_DIR = WORKDIR / "py_train"
PY_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# ---- DataLoader ----
# LeRobotDataset は torch Dataset として使えるので DataLoader が普通に組める
train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE_PY,
    shuffle=True,
    num_workers=TRAIN_NUM_WORKERS_PY,
    drop_last=True,
)

# ---- optimizer: 学習可能な params だけを渡す ----
# ここでは FlowMatchingHead 本体 + vlm_projector + state_proj + pos_embed が対象.
# VLM は freeze してあるので勝手に requires_grad=False で除外される.
_trainable_params = [p for p in smolvla.parameters() if p.requires_grad]
print(f"Optimizer target: {sum(p.numel() for p in _trainable_params)/1e6:.2f}M params")

optimizer = AdamW(_trainable_params, lr=TRAIN_LR_PY, weight_decay=1e-4)


# ---- LR schedule: linear warmup → cosine decay ----
def _lr_lambda(step: int):
    if step < TRAIN_WARMUP_PY:
        # warmup 中は 0 → 1 に線形上昇
        return step / max(TRAIN_WARMUP_PY, 1)
    # warmup 後は cosine で 1.0 → 0.1 (peak lr の 10% まで下げる)
    progress = (step - TRAIN_WARMUP_PY) / max(TRAIN_STEPS_PY - TRAIN_WARMUP_PY, 1)
    progress = min(max(progress, 0.0), 1.0)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return 0.1 + 0.9 * cosine


scheduler = LambdaLR(optimizer, lr_lambda=_lr_lambda)


# ---- 学習ループ ----
smolvla.train()
_step = 0
_loss_history = []
_start = time.time()

print(f"\nTraining {TRAIN_STEPS_PY} steps (batch={TRAIN_BATCH_SIZE_PY})...", flush=True)

while _step < TRAIN_STEPS_PY:
    for batch in train_loader:
        if _step >= TRAIN_STEPS_PY:
            break

        # 画像を PIL/numpy に整形 (RealVLMEncoder が受け付ける形式)
        # LeRobotDataset は (B, C, H, W) float in [0, 1] で返す
        # LIBERO dataset のカメラキーは "front" (俯瞰) と "wrist" (手先).
        # 教材版では front だけ使う (multi-view は改造ネタの一つ)
        imgs_tensor = batch["observation.images.front"]
        if imgs_tensor.dtype.is_floating_point:
            imgs_tensor = (imgs_tensor * 255).clamp(0, 255).to(torch.uint8)
        imgs_np = imgs_tensor.permute(0, 2, 3, 1).cpu().numpy()  # (B, H, W, C)
        imgs_list = [imgs_np[i] for i in range(imgs_np.shape[0])]

        # タスク説明文
        tasks_list = batch["task"]
        if isinstance(tasks_list, str):
            tasks_list = [tasks_list]

        # action を GPU に + 正規化
        actions = batch["action"].to("cuda", dtype=torch.float32)
        actions_norm = normalize_actions(actions)

        # ★ state (現在のロボット状態) を GPU に. LIBERO は (B, 8).
        states = batch["observation.state"].to("cuda", dtype=torch.float32)

        # forward → loss → backward
        loss = smolvla.compute_loss(imgs_list, tasks_list, states, actions_norm)

        optimizer.zero_grad()
        loss.backward()
        # 勾配爆発対策 (max_norm=1.0)
        torch.nn.utils.clip_grad_norm_(_trainable_params, max_norm=1.0)
        optimizer.step()
        scheduler.step()

        _loss_history.append(loss.item())

        # ログ出力: LOG_FREQ step ごとに loss / lr / elapsed を print
        if (_step + 1) % TRAIN_LOG_FREQ_PY == 0 or _step == 0:
            _lr = scheduler.get_last_lr()[0]
            _elapsed = time.time() - _start
            _avg_loss = sum(_loss_history[-TRAIN_LOG_FREQ_PY:]) / min(
                len(_loss_history), TRAIN_LOG_FREQ_PY
            )
            print(
                f"step {_step+1:4d}/{TRAIN_STEPS_PY}  "
                f"loss={_avg_loss:.4f}  lr={_lr:.2e}  "
                f"elapsed={_elapsed:.1f}s",
                flush=True,
            )

        _step += 1

_train_time = time.time() - _start
print(f"\nTraining done in {_train_time:.1f}s ({_train_time/60:.1f} min)")

# ---- Loss curve をグラフ化 (学習が回ったか目視で確認) ----
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(8, 4))
_x = np.arange(1, len(_loss_history) + 1)
_y = np.array(_loss_history)
ax.plot(_x, _y, color="#8888cc", alpha=0.4, linewidth=0.8, label="loss (raw)")

# 移動平均でノイズを smoothing (window = LOG_FREQ)
if len(_y) >= TRAIN_LOG_FREQ_PY:
    _w = TRAIN_LOG_FREQ_PY
    _smooth = np.convolve(_y, np.ones(_w) / _w, mode="valid")
    _x_smooth = _x[_w - 1:]
    ax.plot(_x_smooth, _smooth, color="#2244aa", linewidth=1.5,
            label=f"moving avg (window={_w})")

ax.set_xlabel("step")
ax.set_ylabel("Flow Matching loss (MSE)")
ax.set_title(
    f"Training loss ({len(_loss_history)} steps, "
    f"final={_loss_history[-1]:.4f}, "
    f"min={min(_loss_history):.4f})"
)
ax.grid(True, alpha=0.3)
ax.legend(loc="upper right")
fig.tight_layout()

plt.show()

# ---- checkpoint 保存 (学習対象 module + action stats + hyperparams) ----
# flow_head だけでなく vlm_projector, state_proj も学習対象なので一緒に保存.
# (これらを保存しないと復元時にランダム初期化のまま推論が走る)
_ckpt = {
    "flow_head_state_dict":     smolvla.flow_head.state_dict(),
    "vlm_projector_state_dict": smolvla.vlm_projector.state_dict(),
    "state_proj_state_dict":    smolvla.state_proj.state_dict(),
    "action_mean": action_mean.cpu(),
    "action_std":  action_std.cpu(),
    "chunk_size":        smolvla.chunk_size,
    "action_dim":        smolvla.action_dim,
    "state_dim":         smolvla.state_dim,
    "hidden_dim":        smolvla.hidden_dim,
    "loss_history":      _loss_history,
}
_ckpt_path = PY_CHECKPOINT_DIR / "smolvla_ckpt.pt"
torch.save(_ckpt, _ckpt_path)
print(f"\nSaved checkpoint to {_ckpt_path}")

## 10. 評価と rollout 動画

学習した `smolvla` を LIBERO 環境で動かして成功率を測定 + rollout を **mp4** に保存して notebook に埋め込む。

> **⚠️ 成功率は 0% でも問題ありません**  
> 教材版は from-scratch で 300 step しか学習していないため、実用的な成功率は出ません (むしろ **失敗前提**)。 Basic 課題は「学習→評価が動いた」ことが目的で、成功率は判定に使いません。 動画を見ればロボットがヨタヨタと動いてる様子が確認できるはず — それで OK。

### やること
1. `LiberoEnv` を作って `env.reset()` → `env.step()` のループを回す
2. モデルの `sample_actions()` で action chunk を予測、chunk を消費し終わったら次の chunk を予測
3. 途中で frame をキャプチャして mp4 に保存
4. `terminated=True` or 最大 step 到達で終了、成功か失敗かを記録

### 10.1 LIBERO 環境で rollout (成功率 + フレーム収集)

1 episode ぶんの rollout 関数と、それを複数タスク・複数エピソードに広げる関数を用意。
`collect_frames=True` にすると frame を返す (後で mp4 化する)。

In [ ]:
# ==============================================================
# 10.1 LIBERO 環境で rollout (成功率 + フレーム収集)
# ==============================================================
from lerobot.envs.libero import LiberoEnv
from libero.libero import benchmark as libero_benchmark

# ---- 評価設定 (時間短縮のため 2 task × 1 ep. 本気なら PY_EVAL_EPISODES_PER_TASK を増やす) ----
PY_EVAL_TASK_IDS = [0, 1]              # LIBERO-Spatial の task 0-1 だけ
PY_EVAL_EPISODES_PER_TASK = 1          # 各 task 1 episode. 合計 2 rollout
PY_EVAL_MAX_STEPS = 300                # 1 episode の最大 env step
PY_EVAL_NUM_FLOW_STEPS = 10            # Flow matching の Euler step 数


def rollout_one_episode(model, env, seed: int, collect_frames: bool = False) -> dict:
    """1 episode 実行して結果 (成功可否 + オプションで frame 列) を返す.

    action chunking の仕組み:
      chunk 予測 → chunk_size 個の action を順に env に投げる → 消費し切ったら次の chunk 予測
    """
    model.eval()
    device = next(model.flow_head.parameters()).device

    obs, info = env.reset(seed=seed)
    task_desc = env.task_description  # 例: "pick up the black bowl on the plate"

    chunk_buffer = None    # 現在保持している action chunk (predicted, denormalized numpy)
    chunk_step = 0         # chunk 内で何番目にいるか
    success = False
    total_steps = 0
    frames = [] if collect_frames else None

    # 1 episode 内の env step 数を進捗表示 (rollout 完了で自動的に消える)
    from tqdm.auto import tqdm as _tqdm_inner
    _inner_pbar = _tqdm_inner(
        total=PY_EVAL_MAX_STEPS, desc="  steps", unit="step", leave=False
    )

    while total_steps < PY_EVAL_MAX_STEPS:
        # 現在の観測画像 (front カメラ)
        img = obs["pixels"]["image"]   # (H, W, C) uint8

        # frame をキャプチャ (動画化用)
        if collect_frames:
            # LIBERO は agentview を上下反転して返すので、動画用は普通の向きに戻す
            frames.append(img[::-1, ::-1].copy())

        # ★ 現在の robot state を dataset の observation.state 形式に整形.
        # 本教材で使う LIBERO データセットでは、
        # joints.pos (7,) + gripper.qpos の先頭要素 (1,) = 8 次元として構成する.
        try:
            _joint_pos = obs["robot_state"]["joints"]["pos"]            # (7,) numpy
            _gripper_q = obs["robot_state"]["gripper"]["qpos"]           # (2,) numpy
            _state_vec = np.concatenate(
                [_joint_pos, _gripper_q[:1]], axis=0
            ).astype(np.float32)                                          # (8,)
        except (KeyError, TypeError) as _e:
            # ゼロで埋めて誤魔化さず、原因を明示して停止する.
            # 発生時は LiberoEnv の obs_type="pixels_agent_pos" を確認.
            raise RuntimeError(
                "LIBERO env observation から state を構築できません: "
                f"{_e}. obs.keys={list(obs.keys())}"
            ) from _e
        state_tensor = torch.from_numpy(_state_vec).unsqueeze(0).to(device)  # (1, 8)

        # chunk が空 or 使い切ったら新しい chunk を予測
        if chunk_buffer is None or chunk_step >= chunk_buffer.shape[0]:
            with torch.no_grad():
                pred_normalized = model.sample_actions(
                    [img], [task_desc], state_tensor,
                    num_steps=PY_EVAL_NUM_FLOW_STEPS,
                )  # (1, chunk_size, 7)
            # 正規化空間 → 実 action スケールへ
            pred_denorm = denormalize_actions(pred_normalized[0])   # (chunk_size, 7)
            chunk_buffer = pred_denorm.cpu().numpy()
            chunk_step = 0

        # chunk から次の 1 action を取り出して env に投げる
        action = chunk_buffer[chunk_step]  # (7,)
        chunk_step += 1
        total_steps += 1

        obs, reward, terminated, truncated, info = env.step(action)
        _inner_pbar.update(1)

        if terminated or truncated:
            success = bool(info.get("success", terminated))
            break

    _inner_pbar.close()
    return {
        "success": success,
        "steps": total_steps,
        "frames": frames,   # collect_frames=False なら None
    }


# ---- 全 task × 全 episode を rollout ----
print(
    f"Evaluating custom SmolVLA on {len(PY_EVAL_TASK_IDS)} tasks × "
    f"{PY_EVAL_EPISODES_PER_TASK} episodes...",
    flush=True,
)

# LIBERO benchmark suite を取得
_suite = libero_benchmark.get_benchmark_dict()["libero_spatial"]()

py_eval_results = {"per_task": [], "overall_success": None, "rollouts": []}
_total_success = 0
_total_rollouts = 0

# 全 rollout の進捗バー (外側)
from tqdm.auto import tqdm as _tqdm_outer
_total_expected = len(PY_EVAL_TASK_IDS) * PY_EVAL_EPISODES_PER_TASK
_outer_pbar = _tqdm_outer(total=_total_expected, desc="rollout", unit="ep")

for task_id in PY_EVAL_TASK_IDS:
    _task_success = 0
    for ep_i in range(PY_EVAL_EPISODES_PER_TASK):
        # env を毎 episode 新規作成 (状態を持ち越さない)
        env = LiberoEnv(
            task_suite=_suite,
            task_id=task_id,
            task_suite_name="libero_spatial",
            episode_index=ep_i,
            n_envs=PY_EVAL_EPISODES_PER_TASK,
            observation_height=256,
            observation_width=256,
            camera_name=["agentview_image"],
            camera_name_mapping={"agentview_image": "image"},
            control_mode="relative",
            is_libero_plus=True,
            obs_type="pixels_agent_pos",   # ← state 入力用. これで obs に "robot_state" が入る
        )
        # 最初の episode だけ frame を集めて動画にする (メモリ節約)
        collect = (task_id == PY_EVAL_TASK_IDS[0] and ep_i == 0)
        result = rollout_one_episode(
            smolvla, env, seed=EVAL_SEED + task_id * 100 + ep_i,
            collect_frames=collect,
        )
        env.close()

        _task_success += int(result["success"])
        _total_success += int(result["success"])
        _total_rollouts += 1
        py_eval_results["rollouts"].append({
            "task_id": task_id,
            "episode": ep_i,
            "success": result["success"],
            "steps": result["steps"],
            "frames": result["frames"],
        })
        _outer_pbar.update(1)
        _outer_pbar.set_postfix_str(
            f"task {task_id} ep {ep_i}: "
            f"{'✅ SUCCESS' if result['success'] else '❌ FAIL'} "
            f"({result['steps']} steps)"
        )

    _task_rate = 100.0 * _task_success / PY_EVAL_EPISODES_PER_TASK
    py_eval_results["per_task"].append({
        "task_id": task_id,
        "task": SPATIAL_TASK_NAMES[task_id],
        "success_rate": _task_rate,
        "n_success": _task_success,
        "n_episodes": PY_EVAL_EPISODES_PER_TASK,
    })

_outer_pbar.close()

py_eval_results["overall_success"] = (
    100.0 * _total_success / max(_total_rollouts, 1)
)

print(f"\n=== Custom SmolVLA overall: {py_eval_results['overall_success']:.1f}% ===")

### 10.2 動画に固めて notebook に埋め込む

10.1 で集めた frame 列を **mp4** にエンコードして表示。`av` (PyAV) を使う。

In [ ]:
# ==============================================================
# 10.2 frame 列を mp4 化して埋め込み表示
# ==============================================================
import av
from IPython.display import Video, display

PY_VIDEO_DIR = WORKDIR / "py_videos"
PY_VIDEO_DIR.mkdir(parents=True, exist_ok=True)


def write_mp4(frames: list, path: Path, fps: int = 20) -> None:
    """RGB uint8 の frame 列を H.264 mp4 に書き出す."""
    if not frames:
        raise RuntimeError("frames が空")

    h, w = frames[0].shape[:2]
    # H.264 の yuv420p は幅・高さが偶数である必要がある. 奇数なら 1px 落とす.
    h -= h % 2
    w -= w % 2

    with av.open(str(path), mode="w") as container:
        stream = container.add_stream("libx264", rate=fps)
        stream.width = w
        stream.height = h
        stream.pix_fmt = "yuv420p"
        # crf 小さいほど高画質 & ファイル大. 23 が libx264 の既定
        stream.options = {"crf": "23"}

        for fr in frames:
            fr = fr[:h, :w]                          # 偶数化のため切り詰め
            frame = av.VideoFrame.from_ndarray(fr, format="rgb24")
            for packet in stream.encode(frame):
                container.mux(packet)
        # flush 残りパケット
        for packet in stream.encode():
            container.mux(packet)


# rollout 結果から frames を持つものだけ mp4 化して表示
saved = 0
for r in py_eval_results["rollouts"]:
    if not r["frames"]:
        continue
    mp4_path = PY_VIDEO_DIR / f"task{r['task_id']}_ep{r['episode']}.mp4"
    write_mp4(r["frames"], mp4_path, fps=20)
    label = "SUCCESS" if r["success"] else "FAIL"
    print(f"\n▶ task {r['task_id']} ep {r['episode']}  ({label}, {r['steps']} steps)")
    display(Video(str(mp4_path), embed=True, html_attributes="controls loop"))
    saved += 1

if saved == 0:
    print("(collect_frames=False だった rollout ばかり. section 10.1 で collect フラグを true にしてください)")
else:
    print(f"\nSaved {saved} mp4(s) to {PY_VIDEO_DIR}")

## 11. 提出用ログを書き出す (Basic 課題の要)

`section 8-9` の TODO 1〜4 を **全部正しく埋められた場合のみ** ここまで実行できています
(未実装なら `NotImplementedError` で止まっているはず)。

このセルを実行すると **`submission.json` が生成 + ブラウザに自動ダウンロード**されます。
提出はこの JSON 1 ファイルだけです。

### 生成ファイル

- **`submission.json`** ← ★これだけ提出★
- `submission.md` (参照用の人間可読サマリ、提出不要)
- `pip_freeze.txt` (参照用の環境情報、提出不要)

参照用の 2 つも Drive の `MyDrive/smolvla_task/submission/` にバックアップされます。

**成功率は判定には使いません** (Basic は notebook 完走が目的)。
高得点や コントリビューション賞を狙いたい人は Advanced (別途) に進んでください。

In [ ]:
# ==============================================================
# 11 提出用ログ (Basic 版)
# ==============================================================


import hashlib
import json
import platform
import subprocess
from datetime import datetime, timezone

# ▼ 任意: 何を試したか自由に書き残す (未記入でも OK) ▼
METHODS_NOTES = "(自由記述: どんな工夫を試したか等)"

SUBMISSION_DIR = WORKDIR / "submission"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)


def sha256_of_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


# ---- 実行の証拠 ----
_ckpt_path = PY_CHECKPOINT_DIR / "smolvla_ckpt.pt"
fingerprints = {}
if _ckpt_path.is_file():
    fingerprints["smolvla_ckpt.pt"] = sha256_of_file(_ckpt_path)

# 動画ファイルの有無を確認
_video_paths = sorted(
    (WORKDIR / "py_videos").glob("*.mp4")
)

execution_evidence = {
    "vlm_hidden_dim": int(vlm_encoder.hidden_dim),
    "transformer_hidden_dim": int(smolvla.transformer_hidden_dim),
    "trainable_params": int(sum(
        p.numel() for p in smolvla.parameters() if p.requires_grad
    )),
    "training_steps_completed": int(TRAIN_STEPS_PY),
    "training_final_loss": (
        float(_loss_history[-1]) if _loss_history else None
    ),
    "eval_total_rollouts": int(len(py_eval_results.get("rollouts", []))),
    "eval_success_count": int(sum(
        1 for r in py_eval_results.get("rollouts", []) if r.get("success")
    )),
    "eval_overall_success_percent": float(
        py_eval_results.get("overall_success", 0.0)
    ),
    "video_generated": bool(_video_paths),
    "video_count": len(_video_paths),
}

# ---- 環境情報 ----
try:
    pip_freeze = subprocess.check_output(
        [sys.executable, "-m", "pip", "freeze"], text=True,
    )
except Exception as e:
    pip_freeze = f"# pip freeze failed: {e}"

environment = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda if torch.cuda.is_available() else None,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}

# ---- 提出 JSON を構成 ----
submission = {
    "notebook_version": "basic-v1",
    "submitted_at": datetime.now(timezone.utc).isoformat(),
    "methods_notes": METHODS_NOTES,
    "todos_completed": [
        "TODO_1_time_embedding_outer_product",
        "TODO_2_flow_matching_interpolation",
        "TODO_3_target_velocity",
        "TODO_4_euler_step",
    ],
    "execution_evidence": execution_evidence,
    "model_fingerprint": fingerprints,
    "environment": environment,
    "hyperparams": {
        "chunk_size": smolvla.chunk_size,
        "action_dim": smolvla.action_dim,
        "transformer_hidden_dim": smolvla.transformer_hidden_dim,
        "num_transformer_layers": len(smolvla.flow_head.blocks),
        "train_steps": TRAIN_STEPS_PY,
        "batch_size": TRAIN_BATCH_SIZE_PY,
        "learning_rate": TRAIN_LR_PY,
        "warmup_steps": TRAIN_WARMUP_PY,
        "seed": SEED,
        "num_flow_steps_eval": PY_EVAL_NUM_FLOW_STEPS,
    },
}

# ---- 書き出し ----
sub_json = SUBMISSION_DIR / "submission.json"
sub_json.write_text(
    json.dumps(submission, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

# 人間可読サマリ
_per_task = "\n".join(
    f"- task {r['task_id']} ep {r['episode']}: "
    f"{'✅' if r['success'] else '❌'} ({r['steps']} steps)"
    for r in py_eval_results.get("rollouts", [])
)
sub_md = SUBMISSION_DIR / "submission.md"
sub_md.write_text(
    f"""# Submission

**Submitted**: {submission['submitted_at']}
**Version**: basic-v1

## TODO 完了状況
✅ 全ての TODO を実装済 (notebook がここまで実行できた時点で確定)

## 実行の証拠
- VLM hidden dim: {execution_evidence['vlm_hidden_dim']}
- Transformer hidden dim: {execution_evidence['transformer_hidden_dim']}
- 学習可能パラメタ: {execution_evidence['trainable_params']:,}
- 学習 step: {execution_evidence['training_steps_completed']}
- 最終 loss: {execution_evidence['training_final_loss']}
- Rollout 数: {execution_evidence['eval_total_rollouts']}
- 成功数: {execution_evidence['eval_success_count']}
- 動画生成: {'✅' if execution_evidence['video_generated'] else '❌'}

## Rollout 詳細
{_per_task}

## Notes
{METHODS_NOTES}

## Environment
- {environment['gpu']}
- torch {environment['torch']}, cuda {environment['cuda']}
- python {environment['python']}
""",
    encoding="utf-8",
)

# pip freeze も同梱
(SUBMISSION_DIR / "pip_freeze.txt").write_text(pip_freeze, encoding="utf-8")

# ---- Drive にバックアップ (ランタイム終了で消えないように) ----
# 提出物は submission.json だけだが、参照用に md / pip_freeze も一緒に置いておく.
_drive_dir = DRIVE_BACKUP_DIR / "submission"
_drive_dir.mkdir(parents=True, exist_ok=True)
for f in [sub_json, sub_md, SUBMISSION_DIR / "pip_freeze.txt"]:
    shutil.copy2(f, _drive_dir / f.name)

# ---- submission.json をブラウザにダウンロード ----
# 経験的に、ipywidgets Button の on_click 経由で files.download を呼ぶと
# ブラウザによっては download prompt が出ないことがある.
# セル実行の user gesture がまだ有効な、セル末尾の同期呼び出しの方が確実.
print("✅ 提出用ファイル (submission.json) を生成しました:")
print(f"  {sub_json}")
print()
print("参照用に以下も生成:")
print(f"  {sub_md}")
print(f"  {SUBMISSION_DIR / 'pip_freeze.txt'}")
print()
print(f"✅ Drive にもバックアップ済: {_drive_dir}")
print()

# ブラウザに download を投げる. ここで自動的にダウンロードが始まる.
# もし download prompt が出なければ、Drive から直接取得するのが確実:
#   https://drive.google.com/ → MyDrive → smolvla_task → submission → submission.json
from google.colab import files
try:
    files.download(str(sub_json))
    print("📥 ブラウザに submission.json のダウンロードを送信しました.")
except Exception as _e:
    print(f"⚠️ files.download が失敗: {_e}")

print()
print("=" * 60)
print("💡 download prompt が出ない場合は Drive から取得してください:")
print()
print("   Drive → MyDrive → smolvla_task → submission → submission.json")
print()
print(f"   (Colab パス: {_drive_dir / sub_json.name})")
print("=" * 60)